# Arena Ranker — Kaggle GPU 训练与推理

本 notebook 在 Kaggle 提供的 **GPU 环境（T4 16GB / P100 16GB）** 中完成：

1. 安装依赖
2. 设置项目源码
3. 使用 **QLoRA** 微调 `Qwen/Qwen3.5-0.8B` 偏好分类模型
4. 推理并生成 `submission.csv`

### 核心架构

```
prompt + response_a + response_b
        ↓
  apply_chat_template (system + user)
        ↓
  Qwen3.5-0.8B (4-bit NF4 量化)
        ↓
  AutoModelForSequenceClassification
        ↓
  score (分类头, 3-class logits)
```

> **前置准备**
>
> | 项目 | 操作 |
> |------|------|
> | **竞赛数据** | Notebook 右侧 → Add Input → 搜索并添加竞赛数据集 |
> | **GPU** | Settings → Accelerator → 选择 **GPU T4 ×2** 或 **GPU P100** |
> | **联网** | Settings → Internet → **On**（用于下载 HuggingFace 模型）|
>
> 如果不想联网下载模型，请参考最后一节「离线模式」。


## 1. 安装依赖

Kaggle 环境已预装 PyTorch，这里补装 QLoRA 训练所需的包。

In [1]:
!pip install -q \
    "transformers>=5.0.0" \
    "peft>=0.17.0" \
    "bitsandbytes>=0.45.0" \
    "datasets>=3.0.0" \
    "accelerate>=1.0.0" \
    "scikit-learn>=1.5.0" \
    "tqdm>=4.66.0" \
    "pyyaml>=6.0.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.7 MB/s eta 0:00:00:00:0100:01


## 2. 写入项目源码

将 `arena_ranker` 包的所有源文件写入 `/kaggle/working/arena_ranker/`。

> **替代方案**：也可以把本仓库上传为 Kaggle Dataset，
> 然后 `!pip install /kaggle/input/<your-dataset-slug>/` 来安装。

In [2]:
import base64
from pathlib import Path

PKG_DIR = Path("/kaggle/working/arena_ranker")
PKG_DIR.mkdir(parents=True, exist_ok=True)

_FILES = {
    "__init__.py": "IiIiQXJlbmEgUmFua2VyIOKAlCBRTG9SQSDlvq7osIMgUXdlbjMuNS0wLjhCIOeUqOS6jiBDaGF0Qm90IEFyZW5hIOWBj+WlvemihOa1i+OAgiIiIgo=",
    "config.py": "IiIiCumFjee9ruaooeWdlyDigJQg5a6a5LmJ6K6t57uD5ZKM5o6o55CG5omA6ZyA55qE5YWo6YOo6buY6K6k5Y+C5pWw44CCCgrmnKzmqKHlnZfph4fnlKggZGF0YWNsYXNzIOe7hOe7h+mFjee9ru+8jOaUr+aMgSBZQU1MIOaMgeS5heWMluOAggrkuLvopoHliIbkuLrkuInpg6jliIbvvJoKICAtIERhdGFDb25maWc6ICDmlbDmja7ot6/lvoTjgIHmlofmnKzmiKrmlq3jgIHpqozor4Hpm4bmr5TkvosKICAtIE1vZGVsQ29uZmlnOiDln7rluqfmqKHlnovjgIHph4/ljJbjgIFMb1JBIOWPguaVsAogIC0gVHJhaW5pbmdDb25maWc6IOWtpuS5oOeOh+OAgWJhdGNoIHNpemXjgIFlcG9jaCDnrYkgVHJhaW5lciDlj4LmlbAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCB5YW1sCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5qCH562+55u45YWz5bi46YePCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkxBQkVMX0NPTFVNTlMgPSBbIndpbm5lcl9tb2RlbF9hIiwgIndpbm5lcl9tb2RlbF9iIiwgIndpbm5lcl90aWUiXQpMQUJFTF9UT19JRCA9IHsid2lubmVyX21vZGVsX2EiOiAwLCAid2lubmVyX21vZGVsX2IiOiAxLCAid2lubmVyX3RpZSI6IDJ9CklEX1RPX0xBQkVMID0ge3Y6IGsgZm9yIGssIHYgaW4gTEFCRUxfVE9fSUQuaXRlbXMoKX0KTlVNX0xBQkVMUyA9IDMKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOWvueivneaooeadv+W4uOmHjwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpTWVNURU1fUFJPTVBUID0gIuS9oOaYr+S4gOS4quWFrOato+eahOivhOWnlO+8jOivt+WFqOmdouivhOS8sOS4pOS4quWbnuetlO+8jOW5tumihOa1i+S6uuexu+acgOWBj+WlveeahOmAiemhueOAgiIKClVTRVJfVEVNUExBVEUgPSAoCiAgICAi5Lul5LiL5piv55So5oi355qE5o+Q6Zeu77yaXG57cHJvbXB0fVxuXG4iCiAgICAi5Zue562UQe+8mlxue3Jlc3BvbnNlX2F9XG5cbiIKICAgICLlm57nrZRC77yaXG57cmVzcG9uc2VfYn1cblxuIgogICAgIue7vOWQiOivhOS8sO+8jOS9oOiupOS4uuWTquS4quWbnuetlOabtOWlve+8nyIKKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIERhdGFDb25maWcKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBEYXRhQ29uZmlnOgogICAgdHJhaW5fcGF0aDogc3RyID0gInRyYWluLmNzdiIKICAgIHRlc3RfcGF0aDogc3RyID0gInRlc3QuY3N2IgogICAgdGV4dF9tYXhfY2hhcnM6IGludCA9IDYwMDAKICAgIHZhbGlkYXRpb25fc2l6ZTogZmxvYXQgPSAwLjEKICAgIHJhbmRvbV9zdGF0ZTogaW50ID0gNDIKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBNb2RlbENvbmZpZwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE1vZGVsQ29uZmlnOgogICAgbW9kZWxfbmFtZTogc3RyID0gIlF3ZW4vUXdlbjMuNS0wLjhCIgogICAgY2FjaGVfZGlyOiBzdHIgfCBOb25lID0gTm9uZQogICAgbG9jYWxfZmlsZXNfb25seTogYm9vbCA9IEZhbHNlCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSAxMDI0CgogICAgIyAtLS0gNC1iaXQg6YeP5YyWIChRTG9SQSkgLS0tCiAgICBsb2FkX2luXzRiaXQ6IGJvb2wgPSBUcnVlCiAgICBibmJfNGJpdF9xdWFudF90eXBlOiBzdHIgPSAibmY0IgogICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudDogYm9vbCA9IFRydWUKCiAgICAjIC0tLSBMb1JBIC0tLQogICAgdXNlX2xvcmE6IGJvb2wgPSBUcnVlCiAgICBsb3JhX3I6IGludCA9IDMyCiAgICBsb3JhX2FscGhhOiBpbnQgPSA2NAogICAgbG9yYV9kcm9wb3V0OiBmbG9hdCA9IDAuMDUKICAgIGxvcmFfYmlhczogc3RyID0gIm5vbmUiCiAgICBsb3JhX3RhcmdldF9tb2R1bGVzOiBsaXN0W3N0cl0gPSBmaWVsZCgKICAgICAgICBkZWZhdWx0X2ZhY3Rvcnk9bGFtYmRhOiBbCiAgICAgICAgICAgICJxX3Byb2oiLCAia19wcm9qIiwgInZfcHJvaiIsICJvX3Byb2oiLAogICAgICAgICAgICAiZ2F0ZV9wcm9qIiwgInVwX3Byb2oiLCAiZG93bl9wcm9qIiwKICAgICAgICBdCiAgICApCiAgICBsb3JhX21vZHVsZXNfdG9fc2F2ZTogbGlzdFtzdHJdID0gZmllbGQoCiAgICAgICAgZGVmYXVsdF9mYWN0b3J5PWxhbWJkYTogWyJzY29yZSJdCiAgICApCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAgVHJhaW5pbmdDb25maWcKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBUcmFpbmluZ0NvbmZpZzoKICAgIG91dHB1dF9kaXI6IHN0ciA9ICJhcnRpZmFjdHMvZGVmYXVsdCIKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0ID0gMmUtNAogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDAuMDEKICAgIHBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZTogaW50ID0gMgogICAgcGVyX2RldmljZV9ldmFsX2JhdGNoX3NpemU6IGludCA9IDQKICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwczogaW50ID0gOAogICAgbnVtX3RyYWluX2Vwb2NoczogaW50ID0gMwogICAgd2FybXVwX3N0ZXBzOiBmbG9hdCA9IDAuMQogICAgbHJfc2NoZWR1bGVyX3R5cGU6IHN0ciA9ICJjb3NpbmUiCiAgICBvcHRpbTogc3RyID0gInBhZ2VkX2FkYW13XzMyYml0IgogICAgZnAxNjogYm9vbCA9IFRydWUKICAgIGJmMTY6IGJvb2wgPSBGYWxzZQogICAgZ3JhZGllbnRfY2hlY2twb2ludGluZzogYm9vbCA9IFRydWUKICAgIGxvZ2dpbmdfc3RlcHM6IGludCA9IDUwCiAgICBldmFsX3N0cmF0ZWd5OiBzdHIgPSAiZXBvY2giCiAgICBzYXZlX3N0cmF0ZWd5OiBzdHIgPSAiZXBvY2giCiAgICBzYXZlX3RvdGFsX2xpbWl0OiBpbnQgPSAyCiAgICBsb2FkX2Jlc3RfbW9kZWxfYXRfZW5kOiBib29sID0gVHJ1ZQogICAgbWV0cmljX2Zvcl9iZXN0X21vZGVsOiBzdHIgPSAibG9nX2xvc3MiCiAgICBncmVhdGVyX2lzX2JldHRlcjogYm9vbCA9IEZhbHNlCiAgICBzZWVkOiBpbnQgPSA0MgogICAgcmVwb3J0X3RvOiBzdHIgPSAibm9uZSIKICAgIGRhdGFsb2FkZXJfbnVtX3dvcmtlcnM6IGludCA9IDAKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBBcHBDb25maWcgKOmhtuWxguiBmuWQiCkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBBcHBDb25maWc6CiAgICBkYXRhOiBEYXRhQ29uZmlnID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PURhdGFDb25maWcpCiAgICBtb2RlbDogTW9kZWxDb25maWcgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9TW9kZWxDb25maWcpCiAgICB0cmFpbmluZzogVHJhaW5pbmdDb25maWcgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9VHJhaW5pbmdDb25maWcpCgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgcmV0dXJuIGFzZGljdChzZWxmKQoKICAgIGRlZiBzYXZlKHNlbGYsIHBhdGg6IHN0ciB8IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgb3V0cHV0X3BhdGggPSBQYXRoKHBhdGgpCiAgICAgICAgb3V0cHV0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBvdXRwdXRfcGF0aC53cml0ZV90ZXh0KAogICAgICAgICAgICB5YW1sLnNhZmVfZHVtcChzZWxmLnRvX2RpY3QoKSwgc29ydF9rZXlzPUZhbHNlKSwKICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgICAgICApCgoKZGVmIGxvYWRfY29uZmlnKHBhdGg6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gQXBwQ29uZmlnOgogICAgIiIi5LuOIFlBTUwg5paH5Lu25Yqg6L296YWN572u77ybcGF0aD1Ob25lIOaXtui/lOWbnuWFqOmDqOm7mOiupOWAvOOAgiIiIgogICAgaWYgcGF0aCBpcyBOb25lOgogICAgICAgIHJldHVybiBBcHBDb25maWcoKQogICAgcmF3ID0geWFtbC5zYWZlX2xvYWQoUGF0aChwYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICByZXR1cm4gQXBwQ29uZmlnKAogICAgICAgIGRhdGE9RGF0YUNvbmZpZygqKnJhdy5nZXQoImRhdGEiLCB7fSkpLAogICAgICAgIG1vZGVsPU1vZGVsQ29uZmlnKCoqcmF3LmdldCgibW9kZWwiLCB7fSkpLAogICAgICAgIHRyYWluaW5nPVRyYWluaW5nQ29uZmlnKCoqcmF3LmdldCgidHJhaW5pbmciLCB7fSkpLAogICAgKQo=",
    "data.py": "IiIiCuaVsOaNruWkhOeQhuaooeWdlyDigJQg5Yqg6L29IENTVuOAgeaWh+acrOa4hea0l+OAgWNoYXQgdGVtcGxhdGUgdG9rZW5pemF0aW9u44CBbWV0cmljcyDorqHnrpfjgIIKCuaguOW/g+a1geeoi++8mgogIDEuIOivu+WPliBDU1Yg4oaSIOa4hea0l+aWh+acrOWtl+autSAo6Kej5p6QIEpTT04g5pWw57uEIC8gUHl0aG9uIGxpc3Qg562J5qC85byPKQogIDIuIOS9v+eUqCB0b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgpIOaehOW7uiBzeXN0ZW0gKyB1c2VyIOWvueivnQogIDMuIOmAmui/hyBEYXRhc2V0Lm1hcCgpIOWujOaIkCB0b2tlbml6YXRpb27vvIzovpPlh7ogaW5wdXRfaWRzIC8gYXR0ZW50aW9uX21hc2sgLyBsYWJlbHMKICA0LiDmj5DkvpsgY29tcHV0ZV9tZXRyaWNzKCkg5L6bIFRyYWluZXIg5L2/55SoCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdAppbXBvcnQganNvbgpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIGRhdGFzZXRzIGltcG9ydCBEYXRhc2V0CmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBhY2N1cmFjeV9zY29yZSwgbG9nX2xvc3MKZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdAoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCAoCiAgICBMQUJFTF9DT0xVTU5TLAogICAgTEFCRUxfVE9fSUQsCiAgICBTWVNURU1fUFJPTVBULAogICAgVVNFUl9URU1QTEFURSwKICAgIERhdGFDb25maWcsCikKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDmlofmnKzop6PmnpAgJiDmuIXmtJcKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBfcGFyc2VfY29udmVyc2F0aW9uX2ZpZWxkKHZhbHVlOiBBbnkpIC0+IGxpc3Rbc3RyXToKICAgICIiIgogICAg5bCG5Y6f5aeLIENTViDlrZfmrrXop6PmnpDkuLrlrZfnrKbkuLLliJfooajjgIIKICAgIOaUr+aMgeS4ieenjei+k+WFpeagvOW8j++8mgogICAgICAtIOaZrumAmuWtl+espuS4siDihpIg55u05o6l5YyF5YWl5YiX6KGoCiAgICAgIC0gSlNPTiDmlbDnu4TlrZfnrKbkuLIg4oaSIGpzb24ubG9hZHMg6Kej5p6QCiAgICAgIC0gUHl0aG9uIGxpc3Qg5a2X56ym5LiyIOKGkiBhc3QubGl0ZXJhbF9ldmFsIOino+aekAogICAgIiIiCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KToKICAgICAgICByZXR1cm4gW3N0cihpdGVtKSBmb3IgaXRlbSBpbiB2YWx1ZV0KICAgIGlmIHZhbHVlIGlzIE5vbmUgb3IgKGlzaW5zdGFuY2UodmFsdWUsIGZsb2F0KSBhbmQgcGQuaXNuYSh2YWx1ZSkpOgogICAgICAgIHJldHVybiBbXQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIHN0cik6CiAgICAgICAgcmV0dXJuIFtzdHIodmFsdWUpXQogICAgdGV4dCA9IHZhbHVlLnN0cmlwKCkKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiBbXQogICAgZm9yIHBhcnNlciBpbiAoanNvbi5sb2FkcywgYXN0LmxpdGVyYWxfZXZhbCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwYXJzZWQgPSBwYXJzZXIodGV4dCkKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShwYXJzZWQsIGxpc3QpOgogICAgICAgICAgICAgICAgcmV0dXJuIFtzdHIoaXRlbSkgZm9yIGl0ZW0gaW4gcGFyc2VkXQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgU3ludGF4RXJyb3IpOgogICAgICAgICAgICBjb250aW51ZQogICAgcmV0dXJuIFt0ZXh0XQoKCmRlZiBfbm9ybWFsaXplX3RleHQodmFsdWU6IEFueSwgbWF4X2NoYXJzOiBpbnQpIC0+IHN0cjoKICAgICIiIuWwhuWOn+Wni+Wtl+autei9rOS4uue6r+aWh+acrO+8jOaMiSBtYXhfY2hhcnMg5oiq5pat44CCIiIiCiAgICBjaHVua3MgPSBfcGFyc2VfY29udmVyc2F0aW9uX2ZpZWxkKHZhbHVlKQogICAgdGV4dCA9ICJcbiIuam9pbihjaHVuay5zdHJpcCgpIGZvciBjaHVuayBpbiBjaHVua3MgaWYgc3RyKGNodW5rKS5zdHJpcCgpKQogICAgcmV0dXJuIHRleHRbOm1heF9jaGFyc10KCgpkZWYgX2J1aWxkX2xhYmVsKHJvdzogcGQuU2VyaWVzKSAtPiBpbnQ6CiAgICAiIiLku44gb25lLWhvdCDmoIfnrb7liJcg4oaSIOWNleS4gOaVtOaVsOagh+etviAoMD1B6IOcLCAxPULog5wsIDI95bmz5bGAKeOAgiIiIgogICAgZm9yIGNvbCwgbGFiZWxfaWQgaW4gTEFCRUxfVE9fSUQuaXRlbXMoKToKICAgICAgICBpZiBpbnQocm93W2NvbF0pID09IDE6CiAgICAgICAgICAgIHJldHVybiBsYWJlbF9pZAogICAgcmFpc2UgVmFsdWVFcnJvcihmIuaXoOaViOagh+etvuihjDoge3Jvd1tMQUJFTF9DT0xVTU5TXS50b19kaWN0KCl9IikKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDmlbDmja7liqDovb0gJiDpooTlpITnkIYKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2FkX2FuZF9wcmVwcm9jZXNzKAogICAgY3N2X3BhdGg6IHN0ciwKICAgIG1heF9jaGFyczogaW50ID0gNjAwMCwKICAgIGlzX3RyYWluOiBib29sID0gVHJ1ZSwKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiIKICAgIOWKoOi9vSBDU1Yg5bm26aKE5aSE55CG5paH5pys5a2X5q6144CCCgogICAgUmV0dXJuczoKICAgICAgICBEYXRhRnJhbWXvvIzljIXlkKsgaWQgLyBwcm9tcHRfY2xlYW4gLyByZXNwb25zZV9hX2NsZWFuIC8gcmVzcG9uc2VfYl9jbGVhbgogICAgICAgIOS7peWPiiAo5LuF6K6t57uD6ZuGKSBsYWJlbHMg5YiX44CCCiAgICAiIiIKICAgIGRmID0gcGQucmVhZF9jc3YoY3N2X3BhdGgpCiAgICBkZlsicHJvbXB0X2NsZWFuIl0gPSBkZlsicHJvbXB0Il0ubWFwKGxhbWJkYSB4OiBfbm9ybWFsaXplX3RleHQoeCwgbWF4X2NoYXJzKSkKICAgIGRmWyJyZXNwb25zZV9hX2NsZWFuIl0gPSBkZlsicmVzcG9uc2VfYSJdLm1hcChsYW1iZGEgeDogX25vcm1hbGl6ZV90ZXh0KHgsIG1heF9jaGFycykpCiAgICBkZlsicmVzcG9uc2VfYl9jbGVhbiJdID0gZGZbInJlc3BvbnNlX2IiXS5tYXAobGFtYmRhIHg6IF9ub3JtYWxpemVfdGV4dCh4LCBtYXhfY2hhcnMpKQogICAgaWYgaXNfdHJhaW46CiAgICAgICAgZGZbImxhYmVscyJdID0gZGYuYXBwbHkoX2J1aWxkX2xhYmVsLCBheGlzPTEpCiAgICByZXR1cm4gZGYKCgpkZWYgc3BsaXRfdHJhaW5fdmFsaWQoCiAgICBkZjogcGQuRGF0YUZyYW1lLAogICAgY29uZmlnOiBEYXRhQ29uZmlnLAopIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgcGQuRGF0YUZyYW1lXToKICAgICIiIuaMiSBzdHJhdGlmaWVkIHNwbGl0IOWIkuWIhuiuree7g+mbhuWSjOmqjOivgembhuOAgiIiIgogICAgdHJhaW5fZGYsIHZhbGlkX2RmID0gdHJhaW5fdGVzdF9zcGxpdCgKICAgICAgICBkZiwKICAgICAgICB0ZXN0X3NpemU9Y29uZmlnLnZhbGlkYXRpb25fc2l6ZSwKICAgICAgICByYW5kb21fc3RhdGU9Y29uZmlnLnJhbmRvbV9zdGF0ZSwKICAgICAgICBzdHJhdGlmeT1kZlsibGFiZWxzIl0sCiAgICApCiAgICByZXR1cm4gdHJhaW5fZGYucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSwgdmFsaWRfZGYucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIENoYXQgVGVtcGxhdGUgVG9rZW5pemF0aW9uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2NoYXRfbWVzc2FnZXMocHJvbXB0OiBzdHIsIHJlc3BvbnNlX2E6IHN0ciwgcmVzcG9uc2VfYjogc3RyKSAtPiBsaXN0W2RpY3RdOgogICAgIiIiCiAgICDmnoTlu7rlr7nor53mtojmga/liJfooajvvIznlKjkuo4gYXBwbHlfY2hhdF90ZW1wbGF0ZeOAggogICAgICAtIHN5c3RlbTog6K+E5aeU6KeS6Imy5oyH5LukCiAgICAgIC0gdXNlcjogICBwcm9tcHQgKyByZXNwb25zZV9hICsgcmVzcG9uc2VfYiDnmoTlrozmlbTlhoXlrrkKICAgICIiIgogICAgcmV0dXJuIFsKICAgICAgICB7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU1fUFJPTVBUfSwKICAgICAgICB7CiAgICAgICAgICAgICJyb2xlIjogInVzZXIiLAogICAgICAgICAgICAiY29udGVudCI6IFVTRVJfVEVNUExBVEUuZm9ybWF0KAogICAgICAgICAgICAgICAgcHJvbXB0PXByb21wdCwKICAgICAgICAgICAgICAgIHJlc3BvbnNlX2E9cmVzcG9uc2VfYSwKICAgICAgICAgICAgICAgIHJlc3BvbnNlX2I9cmVzcG9uc2VfYiwKICAgICAgICAgICAgKSwKICAgICAgICB9LAogICAgXQoKCmRlZiBfdG9rZW5pemVfc2luZ2xlKGV4YW1wbGU6IGRpY3QsIHRva2VuaXplciwgbWF4X2xlbmd0aDogaW50KSAtPiBkaWN0OgogICAgIiIiCiAgICDlr7nljZXmnaHmoLfmnKzlgZogdG9rZW5pemF0aW9u77yI5L6bIERhdGFzZXQubWFwIOiwg+eUqO+8ieOAggoKICAgIOatpemqpO+8mgogICAgICAxLiDnlKggYXBwbHlfY2hhdF90ZW1wbGF0ZSDlsIblr7nor53moLzlvI/ljJbkuLrmlofmnKwKICAgICAgMi4g55SoIHRva2VuaXplciDnvJbnoIHlubbmiKrmlq3liLAgbWF4X2xlbmd0aAogICAgIiIiCiAgICBtZXNzYWdlcyA9IF9idWlsZF9jaGF0X21lc3NhZ2VzKAogICAgICAgIGV4YW1wbGVbInByb21wdF9jbGVhbiJdLAogICAgICAgIGV4YW1wbGVbInJlc3BvbnNlX2FfY2xlYW4iXSwKICAgICAgICBleGFtcGxlWyJyZXNwb25zZV9iX2NsZWFuIl0sCiAgICApCiAgICAjIOWFiOW+l+WIsOagvOW8j+WMluaWh+acrO+8jOWGjeWNleeLrCB0b2tlbml6ZSDku6Xnsr7noa7mjqfliLbmiKrmlq0KICAgICMgYWRkX2dlbmVyYXRpb25fcHJvbXB0PUZhbHNlOiDliIbnsbvku7vliqHkuI3pnIDopoEgYXNzaXN0YW50IOinkuiJsuWJjee8gAogICAgdGV4dCA9IHRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgIG1lc3NhZ2VzLCB0b2tlbml6ZT1GYWxzZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PUZhbHNlLAogICAgKQogICAgZW5jb2RlZCA9IHRva2VuaXplcigKICAgICAgICB0ZXh0LAogICAgICAgIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlLCAgIyBjaGF0IHRlbXBsYXRlIOW3sue7j+WMheWQq+aJgOacieeJueauiiB0b2tlbgogICAgKQogICAgcmV0dXJuIGVuY29kZWQKCgpkZWYgYnVpbGRfZGF0YXNldCgKICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICB0b2tlbml6ZXIsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSAxMDI0LAogICAgaXNfdHJhaW46IGJvb2wgPSBUcnVlLAopIC0+IERhdGFzZXQ6CiAgICAiIiIKICAgIOS7jiBEYXRhRnJhbWUg5p6E5bu6IHRva2VuaXplZCBIdWdnaW5nRmFjZSBEYXRhc2V044CCCgogICAg6L6T5Ye65YiX77yI6K6t57uD77yJOiBpbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrLCBsYWJlbHMKICAgIOi+k+WHuuWIl++8iOa1i+ivle+8iTogaW5wdXRfaWRzLCBhdHRlbnRpb25fbWFzawogICAgIiIiCiAgICBjb2xzID0gWyJpZCIsICJwcm9tcHRfY2xlYW4iLCAicmVzcG9uc2VfYV9jbGVhbiIsICJyZXNwb25zZV9iX2NsZWFuIl0KICAgIGlmIGlzX3RyYWluOgogICAgICAgIGNvbHMuYXBwZW5kKCJsYWJlbHMiKQogICAgZGF0YXNldCA9IERhdGFzZXQuZnJvbV9wYW5kYXMoZGZbY29sc10sIHByZXNlcnZlX2luZGV4PUZhbHNlKQoKICAgIGRhdGFzZXQgPSBkYXRhc2V0Lm1hcCgKICAgICAgICBsYW1iZGEgeDogX3Rva2VuaXplX3NpbmdsZSh4LCB0b2tlbml6ZXIsIG1heF9sZW5ndGgpLAogICAgICAgIGRlc2M9IlRva2VuaXppbmciLAogICAgKQoKICAgICMg56e76Zmk5paH5pys5YiX77yM5Y+q5L+d55WZ5qih5Z6L6ZyA6KaB55qE5pWw5YC85YiXCiAgICByZW1vdmVfY29scyA9IFsicHJvbXB0X2NsZWFuIiwgInJlc3BvbnNlX2FfY2xlYW4iLCAicmVzcG9uc2VfYl9jbGVhbiIsICJpZCJdCiAgICBkYXRhc2V0ID0gZGF0YXNldC5yZW1vdmVfY29sdW1ucygKICAgICAgICBbYyBmb3IgYyBpbiByZW1vdmVfY29scyBpZiBjIGluIGRhdGFzZXQuY29sdW1uX25hbWVzXQogICAgKQogICAgcmV0dXJuIGRhdGFzZXQKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBNZXRyaWNzCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX3NvZnRtYXgoeDogbnAubmRhcnJheSwgYXhpczogaW50ID0gLTEpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiLmlbDlgLznqLPlrprnmoQgc29mdG1heCDlrp7njrDvvIjpgb/lhY3lvJXlhaUgc2NpcHkg5L6d6LWW77yJ44CCIiIiCiAgICBlX3ggPSBucC5leHAoeCAtIG5wLm1heCh4LCBheGlzPWF4aXMsIGtlZXBkaW1zPVRydWUpKQogICAgcmV0dXJuIGVfeCAvIGVfeC5zdW0oYXhpcz1heGlzLCBrZWVwZGltcz1UcnVlKQoKCmRlZiBjb21wdXRlX21ldHJpY3MoZXZhbF9wcmVkKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgIiIiCiAgICBUcmFpbmVyIOeahCBjb21wdXRlX21ldHJpY3Mg5Zue6LCD44CCCgogICAg6K6h566XOgogICAgICAtIGxvZ19sb3NzOiDmpoLnjofnu4/oo4HliarlkI7orqHnrpfvvIzpgb/lhY0gbG9nKDApIOWvvOiHtOaegeerr+WAvAogICAgICAtIGFjY3VyYWN5OiDliIbnsbvlh4bnoa7njocKICAgICIiIgogICAgbG9naXRzLCBsYWJlbHMgPSBldmFsX3ByZWQKCiAgICAjIOWkhOeQhuWPr+iDveeahCBOYU7vvIhDUFUg5qih5byP5oiW5pWw5YC85LiN56iz5a6a5pe25Y+v6IO95Ye6546w77yJCiAgICBuYW5fbWFzayA9IG5wLmlzbmFuKGxvZ2l0cykuYW55KGF4aXM9LTEpCiAgICBpZiBuYW5fbWFzay5hbnkoKToKICAgICAgICBsb2dpdHMgPSBucC5jb3B5KGxvZ2l0cykKICAgICAgICBsb2dpdHNbbmFuX21hc2tdID0gMC4wICAjIE5hTiDooYzmm7/mjaLkuLrlnYfljIDliIbluIMKCiAgICAjIGxvZ2l0cyDihpIg5qaC546HCiAgICBwcm9icyA9IF9zb2Z0bWF4KGxvZ2l0cywgYXhpcz0tMSkKCiAgICAjIOamgueOh+ijgeWJqiAo5a+5IGxvZ19sb3NzIOivhOWIhumdnuW4uOmHjeimgSkKICAgIGVwcyA9IDFlLTcKICAgIHByb2JzID0gbnAuY2xpcChwcm9icywgZXBzLCAxLjAgLSBlcHMpCiAgICBwcm9icyA9IHByb2JzIC8gcHJvYnMuc3VtKGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKCiAgICBsb2dsb3NzID0gZmxvYXQobG9nX2xvc3MobGFiZWxzLCBwcm9icywgbGFiZWxzPVswLCAxLCAyXSkpCiAgICBwcmVkcyA9IG5wLmFyZ21heChsb2dpdHMsIGF4aXM9LTEpCiAgICBhY2MgPSBmbG9hdChhY2N1cmFjeV9zY29yZShsYWJlbHMsIHByZWRzKSkKCiAgICByZXR1cm4geyJsb2dfbG9zcyI6IGxvZ2xvc3MsICJhY2N1cmFjeSI6IGFjY30K",
    "hf.py": "IiIiCuaooeWei+WKoOi9veaooeWdlyDigJQg6LSf6LSjIHRva2VuaXplciDlkowgUUxvUkEg5YiG57G75qih5Z6L55qE5Yqg6L2944CCCgrmoLjlv4PnrZbnlaXvvJoKICAxLiBUb2tlbml6ZXI6IOWKoOi9veWQjuajgOafpSBwYWRfdG9rZW7vvIzoi6XnvLrlpLHliJnorr7kuLogZW9zX3Rva2VuCiAgMi4g5qih5Z6LOiDpgJrov4cgQml0c0FuZEJ5dGVzQ29uZmlnIOi/m+ihjCA0LWJpdCDph4/ljJbliqDovb0KICAzLiBMb1JBOiDkvb/nlKggUEVGVCDnmoQgTG9yYUNvbmZpZyDms6jlhaUgYWRhcHRlcu+8jOWQjOaXtuS/neWtmOWIhuexu+WktCAoc2NvcmUpCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKZnJvbSBwZWZ0IGltcG9ydCBMb3JhQ29uZmlnLCBUYXNrVHlwZSwgZ2V0X3BlZnRfbW9kZWwsIHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0ICgKICAgIEF1dG9Db25maWcsCiAgICBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLAogICAgQXV0b1Rva2VuaXplciwKICAgIEJpdHNBbmRCeXRlc0NvbmZpZywKKQoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBNb2RlbENvbmZpZywgTlVNX0xBQkVMUwoKTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyZW5hX3Jhbmtlci5oZiIpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg6L6F5Yqp5Ye95pWwCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2Rlc2NyaWJlX21vZGVsX3NvdXJjZShtb2RlbF9uYW1lOiBzdHIpIC0+IHN0cjoKICAgICIiIuaPj+i/sOaooeWei+adpea6kO+8jOaWueS+v+aOkuafpeWKoOi9vemXrumimOOAgiIiIgogICAgc291cmNlX3BhdGggPSBQYXRoKG1vZGVsX25hbWUpLmV4cGFuZHVzZXIoKQogICAgaWYgc291cmNlX3BhdGguZXhpc3RzKCk6CiAgICAgICAgY29uZmlnX3BhdGggPSBzb3VyY2VfcGF0aCAvICJjb25maWcuanNvbiIKICAgICAgICBpZiBjb25maWdfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmV0dXJuIGYi5qOA5rWL5Yiw5pys5Zyw5qih5Z6L55uu5b2V77yae3NvdXJjZV9wYXRofSIKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmIuajgOa1i+WIsOWQjOWQjeacrOWcsOebruW9lSB7c291cmNlX3BhdGh977yM5L2G57y65bCRIGNvbmZpZy5qc29u44CCIgogICAgICAgICAgICAidHJhbnNmb3JtZXJzIOS8muaKiuWug+W9k+aIkOacrOWcsOaooeWei+i3r+W+hOW5tuebtOaOpeWKoOi9veWksei0peOAgiIKICAgICAgICApCiAgICBpZiAiLyIgaW4gbW9kZWxfbmFtZToKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmImB7bW9kZWxfbmFtZX1gIOWwhuiiq+W9k+S9nCBIdWdnaW5nRmFjZSDku5PlupMgSUTjgIIiCiAgICAgICAgICAgICLpppbmrKHov5DooYzpnIDogZTnvZHkuIvovb3vvJvnprvnur/njq/looPor7fpooTkuIvovb3liLDmnKzlnLDlho3kv67mlLkgbW9kZWxfbmFtZeOAgiIKICAgICAgICApCiAgICByZXR1cm4gZiJge21vZGVsX25hbWV9YCDml6LkuI3mmK/mnKzlnLDnm67lvZXvvIzkuZ/kuI3mmK/moIflh4YgSHVnZ2luZ0ZhY2Ug5LuT5bqTIElE44CCIgoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIFRva2VuaXplcgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIGxvYWRfdG9rZW5pemVyKGNvbmZpZzogTW9kZWxDb25maWcpOgogICAgIiIiCiAgICDliqDovb0gdG9rZW5pemVyIOW5tumFjee9riBwYWRfdG9rZW7jgIIKCiAgICBRd2VuIOezu+WIl+aooeWei+mAmuW4uOayoeaciem7mOiupCBwYWRfdG9rZW7vvIzov5nph4zlsIblhbborr7kuLogZW9zX3Rva2Vu44CCCiAgICDlkIzml7borr7nva4gcGFkZGluZ19zaWRlPSJsZWZ0Iu+8iGRlY29kZXItb25seSDmqKHlnovlgZrliIbnsbvml7bnmoTmjqjojZDlgZrms5XvvInjgIIKICAgICIiIgogICAgdHJ5OgogICAgICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBjb25maWcubW9kZWxfbmFtZSwKICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgY2FjaGVfZGlyPWNvbmZpZy5jYWNoZV9kaXIsCiAgICAgICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9Y29uZmlnLmxvY2FsX2ZpbGVzX29ubHksCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb0gdG9rZW5pemVyIOWksei0pToge2V4Y31cbiIKICAgICAgICAgICAgZiJ7X2Rlc2NyaWJlX21vZGVsX3NvdXJjZShjb25maWcubW9kZWxfbmFtZSl9IgogICAgICAgICkgZnJvbSBleGMKCiAgICAjIOiuvue9riBwYWRfdG9rZW7vvIhRd2VuIOmAmuW4uOayoeaciem7mOiupOWAvO+8iQogICAgaWYgdG9rZW5pemVyLnBhZF90b2tlbiBpcyBOb25lOgogICAgICAgIHRva2VuaXplci5wYWRfdG9rZW4gPSB0b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBMT0dHRVIuaW5mbygKICAgICAgICAgICAgIlRva2VuaXplciDnvLrlsJEgcGFkX3Rva2Vu77yM5bey6K6+5Li6IGVvc190b2tlbjogJyVzJyAoaWQ9JXMpIiwKICAgICAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbiwKICAgICAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCwKICAgICAgICApCgogICAgIyBkZWNvZGVyLW9ubHkg5qih5Z6L5YGa5YiG57G75pe277yM5bem5L6n5aGr5YWF5Y+v6YG/5YWN5pyA5ZCO5LiA5LiqIHRva2VuIOS4uiBwYWQKICAgIHRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKCiAgICByZXR1cm4gdG9rZW5pemVyCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg6YeP5YyW6YWN572uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2JuYl9jb25maWcoY29uZmlnOiBNb2RlbENvbmZpZykgLT4gQml0c0FuZEJ5dGVzQ29uZmlnIHwgTm9uZToKICAgICIiIuaehOW7uiBCaXRzQW5kQnl0ZXNDb25maWcg55So5LqOIDQtYml0IFFMb1JBIOmHj+WMluOAgiIiIgogICAgaWYgbm90IGNvbmZpZy5sb2FkX2luXzRiaXQ6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgICMgNC1iaXQg6YeP5YyW6ZyA6KaBIENVREHvvJtDUFUg5qih5byP5LiL6Lez6L+HCiAgICBpZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBMT0dHRVIud2FybmluZygi5pyq5qOA5rWL5YiwIENVREHvvIzot7Pov4cgNC1iaXQg6YeP5YyW77yI5bCG5LulIGZsb2F0MzIg5Yqg6L295qih5Z6L77yJIikKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPWNvbmZpZy5ibmJfNGJpdF9xdWFudF90eXBlLAogICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICBibmJfNGJpdF91c2VfZG91YmxlX3F1YW50PWNvbmZpZy5ibmJfNGJpdF91c2VfZG91YmxlX3F1YW50LAogICAgKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIExvUkEg6YWN572uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2xvcmFfY29uZmlnKGNvbmZpZzogTW9kZWxDb25maWcpIC0+IExvcmFDb25maWc6CiAgICAiIiIKICAgIOaehOW7uiBMb1JBIOmAgumFjeWZqOmFjee9ruOAggoKICAgIOWFs+mUruWPguaVsOivtOaYjjoKICAgICAgLSB0YXJnZXRfbW9kdWxlczog5a+55omA5pyJIGF0dGVudGlvbiArIEZGTiDmipXlvbHlsYLms6jlhaUgTG9SQQogICAgICAtIG1vZHVsZXNfdG9fc2F2ZT1bInNjb3JlIl06IOWIhuexu+WktCAoc2NvcmUpIOW/hemhu+WFqOmHj+iuree7g+W5tuS/neWtmO+8jAogICAgICAgIOWQpuWImemaj+acuuWIneWni+WMlueahOWIhuexu+WktOS4jeS8muiiq+aMgeS5heWMlgogICAgICAtIHRhc2tfdHlwZT1TRVFfQ0xTOiDlkYror4kgUEVGVCDov5nmmK/kuIDkuKrluo/liJfliIbnsbvku7vliqEKICAgICIiIgogICAgcmV0dXJuIExvcmFDb25maWcoCiAgICAgICAgcj1jb25maWcubG9yYV9yLAogICAgICAgIGxvcmFfYWxwaGE9Y29uZmlnLmxvcmFfYWxwaGEsCiAgICAgICAgdGFyZ2V0X21vZHVsZXM9Y29uZmlnLmxvcmFfdGFyZ2V0X21vZHVsZXMsCiAgICAgICAgbG9yYV9kcm9wb3V0PWNvbmZpZy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgYmlhcz1jb25maWcubG9yYV9iaWFzLAogICAgICAgIHRhc2tfdHlwZT1UYXNrVHlwZS5TRVFfQ0xTLAogICAgICAgIG1vZHVsZXNfdG9fc2F2ZT1jb25maWcubG9yYV9tb2R1bGVzX3RvX3NhdmUsCiAgICApCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5Yqg6L295YiG57G75qih5Z6LIChRTG9SQSkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2FkX21vZGVsKGNvbmZpZzogTW9kZWxDb25maWcsIHRva2VuaXplcj1Ob25lKToKICAgICIiIgogICAg5Yqg6L29IFFMb1JBIOWIhuexu+aooeWei++8jOWujOaVtOa1geeoi++8mgogICAgICAxLiDkvb/nlKggQml0c0FuZEJ5dGVzQ29uZmlnIOi/m+ihjCA0LWJpdCDph4/ljJbliqDovb0KICAgICAgMi4g6YCa6L+HIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24g5pu/5o2i6K+t6KiA5bu65qih5aS05Li65YiG57G75aS0IChudW1fbGFiZWxzPTMpCiAgICAgIDMuIOS9v+eUqCBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nIOWHhuWkh+mHj+WMluaooeWeiwogICAgICA0LiDms6jlhaUgTG9SQSBhZGFwdGVyCgogICAgUmV0dXJuczoKICAgICAgICBQRUZUIOWMheijheWQjueahOaooeWei++8iOiLpSB1c2VfbG9yYT1UcnVl77yJ77yM5ZCm5YiZ5Y6f5aeL5qih5Z6L44CCCiAgICAiIiIKICAgIGJuYl9jb25maWcgPSBfYnVpbGRfYm5iX2NvbmZpZyhjb25maWcpCgogICAgIyBRd2VuMy41IOaYryBWTE0g5qih5Z6L77yMY29uZmlnIOacieW1jOWll+eahCB0ZXh0X2NvbmZpZ+OAggogICAgIyDpnIDopoHnoa7kv50gbnVtX2xhYmVscyDooqvmraPnoa7kvKDmkq3liLAgdGV4dF9jb25maWfjgIIKICAgIHRyeToKICAgICAgICBtb2RlbF9jb25maWcgPSBBdXRvQ29uZmlnLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgY29uZmlnLm1vZGVsX25hbWUsCiAgICAgICAgICAgIG51bV9sYWJlbHM9TlVNX0xBQkVMUywKICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgY2FjaGVfZGlyPWNvbmZpZy5jYWNoZV9kaXIsCiAgICAgICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9Y29uZmlnLmxvY2FsX2ZpbGVzX29ubHksCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb3mqKHlnovphY3nva7lpLHotKU6IHtleGN9XG4iCiAgICAgICAgICAgIGYie19kZXNjcmliZV9tb2RlbF9zb3VyY2UoY29uZmlnLm1vZGVsX25hbWUpfSIKICAgICAgICApIGZyb20gZXhjCgogICAgIyBRd2VuMy41IFZMTSDnmoQgbnVtX2xhYmVscyDlj6/og73msqHmnInkvKDmkq3liLAgdGV4dF9jb25maWcKICAgIGlmIGhhc2F0dHIobW9kZWxfY29uZmlnLCAidGV4dF9jb25maWciKToKICAgICAgICBtb2RlbF9jb25maWcudGV4dF9jb25maWcubnVtX2xhYmVscyA9IE5VTV9MQUJFTFMKICAgICAgICBMT0dHRVIuaW5mbygi5bey5omL5Yqo5bCGIG51bV9sYWJlbHM9JXMg5Lyg5pKt5YiwIHRleHRfY29uZmlnIiwgTlVNX0xBQkVMUykKCiAgICBkdHlwZSA9IHRvcmNoLmZsb2F0MTYgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIHRvcmNoLmZsb2F0MzIKICAgIHRyeToKICAgICAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBjb25maWcubW9kZWxfbmFtZSwKICAgICAgICAgICAgY29uZmlnPW1vZGVsX2NvbmZpZywKICAgICAgICAgICAgcXVhbnRpemF0aW9uX2NvbmZpZz1ibmJfY29uZmlnLAogICAgICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgICAgICBjYWNoZV9kaXI9Y29uZmlnLmNhY2hlX2RpciwKICAgICAgICAgICAgbG9jYWxfZmlsZXNfb25seT1jb25maWcubG9jYWxfZmlsZXNfb25seSwKICAgICAgICAgICAgdG9yY2hfZHR5cGU9ZHR5cGUsCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb3mqKHlnovlpLHotKU6IHtleGN9XG4iCiAgICAgICAgICAgIGYie19kZXNjcmliZV9tb2RlbF9zb3VyY2UoY29uZmlnLm1vZGVsX25hbWUpfSIKICAgICAgICApIGZyb20gZXhjCgogICAgIyDlr7npvZAgcGFkX3Rva2VuX2lkCiAgICBpZiB0b2tlbml6ZXIgaXMgbm90IE5vbmUgYW5kIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwuY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgICAgICBpZiBoYXNhdHRyKG1vZGVsLmNvbmZpZywgInRleHRfY29uZmlnIik6CiAgICAgICAgICAgIG1vZGVsLmNvbmZpZy50ZXh0X2NvbmZpZy5wYWRfdG9rZW5faWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCgogICAgaWYgbm90IGNvbmZpZy51c2VfbG9yYToKICAgICAgICByZXR1cm4gbW9kZWwKCiAgICAjIFFMb1JBIOeJueacieatpemqpDog5YeG5aSH6YeP5YyW5qih5Z6L5Lul6YCC6YWN6K6t57uD77yI5LuF5Zyo5a6e6ZmF6YeP5YyW5pe277yJCiAgICBpZiBjb25maWcubG9hZF9pbl80Yml0IGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIG1vZGVsID0gcHJlcGFyZV9tb2RlbF9mb3Jfa2JpdF90cmFpbmluZygKICAgICAgICAgICAgbW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUKICAgICAgICApCgogICAgbG9yYV9jb25maWcgPSBfYnVpbGRfbG9yYV9jb25maWcoY29uZmlnKQogICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChtb2RlbCwgbG9yYV9jb25maWcpCgogICAgIyDmiZPljbDlj6/orq3nu4Plj4LmlbDnu5/orqEKICAgIG1vZGVsLnByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKCkKCiAgICByZXR1cm4gbW9kZWwK",
    "modeling.py": "IiIiCuaooeWei+WumuS5ieaooeWdl+OAggoK5b2T5YmN54mI5pys5L2/55SoIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24gKyBRTG9SQe+8jArmqKHlnovnmoTliqDovb3lkozphY3nva7lt7Lnp7voh7MgaGYucHnjgILmraTmqKHlnZfkv53nlZnkuLrljaDkvY3vvIzkvpvmnKrmnaXmianlsZXkvb/nlKjjgIIKIiIiCg==",
    "dummy_data.py": "IiIiCuWBh+aVsOaNrueUn+aIkOaooeWdlyDigJQg55Sf5oiQIGR1bW15IHRyYWluLmNzdiDlkowgdGVzdC5jc3Yg55So5LqO5pys5Zyw5rWL6K+V44CCCgrkvb/nlKjmlrnms5XvvJoKICB1diBydW4gYXJlbmEtZHVtbXktZGF0YSAgICAgICAgICAgICAgICAgICAgIyDpu5jorqQgMTAwIOadoeiuree7gyArIDIwIOadoea1i+ivlQogIHV2IHJ1biBhcmVuYS1kdW1teS1kYXRhIC0tbi10cmFpbiA1MCAtLW4tdGVzdCAxMAogIHV2IHJ1biBhcmVuYS1kdW1teS1kYXRhIC0tb3V0cHV0LWRpciAuL2RhdGEKCuS5n+WPr+S7peWcqCBQeXRob24g5Lit55u05o6l6LCD55So77yaCiAgZnJvbSBhcmVuYV9yYW5rZXIuZHVtbXlfZGF0YSBpbXBvcnQgZ2VuZXJhdGVfZHVtbXlfZGF0YQogIGdlbmVyYXRlX2R1bW15X2RhdGEobl90cmFpbj0xMDAsIG5fdGVzdD0yMCkKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHJhbmRvbQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgojIOWBh+aPkOmXruWIl+ihqApfUFJPTVBUUyA9IFsKICAgICJXaGF0IGlzIG1hY2hpbmUgbGVhcm5pbmc/IEV4cGxhaW4gaXQgaW4gc2ltcGxlIHRlcm1zLiIsCiAgICAiV3JpdGUgYSBQeXRob24gZnVuY3Rpb24gdGhhdCBzb3J0cyBhIGxpc3QgdXNpbmcgbWVyZ2Ugc29ydC4iLAogICAgIkV4cGxhaW4gdGhlIHRoZW9yeSBvZiByZWxhdGl2aXR5IHRvIGEgMTAteWVhci1vbGQuIiwKICAgICJIb3cgZG9lcyBwaG90b3N5bnRoZXNpcyB3b3JrPyBQbGVhc2UgYmUgZGV0YWlsZWQuIiwKICAgICJXaGF0IGFyZSB0aGUgbWFpbiBkaWZmZXJlbmNlcyBiZXR3ZWVuIFRDUCBhbmQgVURQPyIsCiAgICAiU3VtbWFyaXplIHRoZSBwbG90IG9mIFJvbWVvIGFuZCBKdWxpZXQuIiwKICAgICJXcml0ZSBhIHBvZW0gYWJvdXQgdGhlIG9jZWFuIGF0IHN1bnNldC4iLAogICAgIkV4cGxhaW4gcXVhbnR1bSBlbnRhbmdsZW1lbnQgaW4gc2ltcGxlIHRlcm1zLiIsCiAgICAiV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBGcmFuY2UgYW5kIHdoYXQgaXMgaXQga25vd24gZm9yPyIsCiAgICAiSG93IGRvIG5ldXJhbCBuZXR3b3JrcyBsZWFybj8gRXhwbGFpbiBiYWNrcHJvcGFnYXRpb24uIiwKICAgICJDb21wYXJlIFB5dGhvbiBhbmQgUnVzdCBmb3Igc3lzdGVtcyBwcm9ncmFtbWluZy4iLAogICAgIldoYXQgYXJlIHRoZSBiZW5lZml0cyBvZiBtZWRpdGF0aW9uPyIsCiAgICAiRGVzY3JpYmUgaG93IGEgQ1BVIGV4ZWN1dGVzIGluc3RydWN0aW9ucy4iLAogICAgIldyaXRlIGEgU1FMIHF1ZXJ5IHRvIGZpbmQgZHVwbGljYXRlIGVtYWlscyBpbiBhIHRhYmxlLiIsCiAgICAiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIHN1cGVydmlzZWQgYW5kIHVuc3VwZXJ2aXNlZCBsZWFybmluZy4iLApdCgpfUkVTUE9OU0VfVEVNUExBVEVTX0EgPSBbCiAgICAiVGhhdCdzIGEgZ3JlYXQgcXVlc3Rpb24hIExldCBtZSBleHBsYWluLiB7dG9waWN9IGlzIGZ1bmRhbWVudGFsbHkgYWJvdXQgIgogICAgInVuZGVyc3RhbmRpbmcgcGF0dGVybnMgaW4gZGF0YS4gVGhlIGtleSBpbnNpZ2h0IGlzIHRoYXQgd2UgY2FuIHVzZSBtYXRoZW1hdGljYWwgIgogICAgIm1vZGVscyB0byBhcHByb3hpbWF0ZSBjb21wbGV4IHJlbGF0aW9uc2hpcHMuIEhlcmUncyBhIG1vcmUgZGV0YWlsZWQgYnJlYWtkb3duOiAiCiAgICAiRmlyc3QsIHdlIG5lZWQgdG8gY29sbGVjdCByZWxldmFudCBkYXRhLiBUaGVuLCB3ZSBwcmVwcm9jZXNzIGl0IHRvIHJlbW92ZSBub2lzZS4gIgogICAgIkZpbmFsbHksIHdlIHRyYWluIG91ciBtb2RlbCBhbmQgZXZhbHVhdGUgaXRzIHBlcmZvcm1hbmNlLiIsCiAgICAiU3VyZSEgSGVyZSdzIG15IHRha2Ugb24gdGhpcy4ge3RvcGljfSBpbnZvbHZlcyBzZXZlcmFsIGltcG9ydGFudCBjb25jZXB0cy4gIgogICAgIlRoZSBtb3N0IGZ1bmRhbWVudGFsIG9uZSBpcyB0aGF0IGxlYXJuaW5nIGhhcHBlbnMgdGhyb3VnaCBpdGVyYXRpdmUgb3B0aW1pemF0aW9uLiAiCiAgICAiV2Ugc3RhcnQgd2l0aCBhIHJhbmRvbSBndWVzcyBhbmQgZ3JhZHVhbGx5IGltcHJvdmUgaXQgYmFzZWQgb24gZmVlZGJhY2sgZnJvbSB0aGUgZGF0YS4iLAogICAgIkdyZWF0IHF1ZXN0aW9uLiBJbiBzaW1wbGUgdGVybXMsIHt0b3BpY30gaXMgbGlrZSB0ZWFjaGluZyBhIGNvbXB1dGVyIHRvIHJlY29nbml6ZSAiCiAgICAicGF0dGVybnMuIEltYWdpbmUgc2hvd2luZyBhIGNoaWxkIHRob3VzYW5kcyBvZiBwaWN0dXJlcyBvZiBjYXRzIGFuZCBkb2dzIC0gZXZlbnR1YWxseSAiCiAgICAidGhleSBsZWFybiB0byB0ZWxsIHRoZW0gYXBhcnQuIFRoYXQncyBlc3NlbnRpYWxseSB3aGF0IGhhcHBlbnMgaW4gdGhpcyBwcm9jZXNzLiIsCl0KCl9SRVNQT05TRV9URU1QTEFURVNfQiA9IFsKICAgICJUaGFua3MgZm9yIGFza2luZyEge3RvcGljfSBpcyBhY3R1YWxseSBzaW1wbGVyIHRoYW4gbW9zdCBwZW9wbGUgdGhpbmsuICIKICAgICJBdCBpdHMgY29yZSwgaXQncyBhYm91dCBmaW5kaW5nIHRoZSBiZXN0IGZ1bmN0aW9uIHRoYXQgbWFwcyBpbnB1dHMgdG8gb3V0cHV0cy4gIgogICAgIldlIGRvIHRoaXMgYnkgbWluaW1pemluZyBhIGxvc3MgZnVuY3Rpb24gdXNpbmcgZ3JhZGllbnQtYmFzZWQgb3B0aW1pemF0aW9uLiAiCiAgICAiVGhlIGJlYXV0eSBvZiB0aGlzIGFwcHJvYWNoIGlzIGl0cyBnZW5lcmFsaXR5IC0gaXQgd29ya3MgYWNyb3NzIG1hbnkgZG9tYWlucy4iLAogICAgIkxldCBtZSBicmVhayB0aGlzIGRvd24uIHt0b3BpY30gY2FuIGJlIHVuZGVyc3Rvb2QgdGhyb3VnaCBhIHNpbXBsZSBhbmFsb2d5LiAiCiAgICAiVGhpbmsgb2YgaXQgYXMgYSByZWNpcGUgLSB5b3UgaGF2ZSBpbmdyZWRpZW50cyAoZGF0YSksIGluc3RydWN0aW9ucyAoYWxnb3JpdGhtKSwgIgogICAgImFuZCBhIGZpbmFsIGRpc2ggKHByZWRpY3Rpb25zKS4gVGhlIHF1YWxpdHkgb2YgZWFjaCBpbmdyZWRpZW50IG1hdHRlcnMsIGJ1dCBzbyBkb2VzICIKICAgICJob3cgeW91IGNvbWJpbmUgdGhlbS4iLAogICAgIkknZCBiZSBoYXBweSB0byBleHBsYWluLiB7dG9waWN9IGlzIGEgZmFzY2luYXRpbmcgYXJlYS4gVGhlIGJhc2ljIGlkZWEgaXMgIgogICAgInRoYXQgd2UgY2FuIHVzZSBzdGF0aXN0aWNzIGFuZCBjb21wdXRhdGlvbiB0byBtYWtlIHByZWRpY3Rpb25zIGFib3V0IHRoZSB3b3JsZC4gIgogICAgIlRoZSBrZXkgY2hhbGxlbmdlIGlzIGdlbmVyYWxpemF0aW9uIC0gbWFraW5nIHN1cmUgb3VyIG1vZGVsIHdvcmtzIG9uIG5ldywgdW5zZWVuIGRhdGEuIiwKXQoKCmRlZiBnZW5lcmF0ZV9kdW1teV9kYXRhKAogICAgb3V0cHV0X2Rpcjogc3RyID0gIi4iLAogICAgbl90cmFpbjogaW50ID0gMTAwLAogICAgbl90ZXN0OiBpbnQgPSAyMCwKICAgIHNlZWQ6IGludCA9IDQyLAopIC0+IHR1cGxlW1BhdGgsIFBhdGhdOgogICAgIiIiCiAgICDnlJ/miJDnlKjkuo7mnKzlnLDmtYvor5XnmoTlgYfmlbDmja7jgIIKCiAgICDmoIfnrb7liIbluIPlpKfoh7TkuLrvvJpB6IOcIDQwJSwgQuiDnCA0MCUsIOW5s+WxgCAyMCXvvIjmqKHmi5/nnJ/lrp7mlbDmja7liIbluIPvvInjgIIKCiAgICBSZXR1cm5zOgogICAgICAgICh0cmFpbl9wYXRoLCB0ZXN0X3BhdGgpCiAgICAiIiIKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZShzZWVkKQogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG91dCA9IFBhdGgob3V0cHV0X2RpcikKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyAtLS0tIOiuree7g+mbhiAtLS0tCiAgICB0cmFpbl9yb3dzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG5fdHJhaW4pOgogICAgICAgIHByb21wdCA9IF9QUk9NUFRTW2kgJSBsZW4oX1BST01QVFMpXQogICAgICAgIHRvcGljID0gcHJvbXB0LnNwbGl0KCI/IilbMF0uc3BsaXQoIi4iKVswXS5zdHJpcCgpCgogICAgICAgIHJlc3BfYSA9IHJhbmRvbS5jaG9pY2UoX1JFU1BPTlNFX1RFTVBMQVRFU19BKS5mb3JtYXQodG9waWM9dG9waWMpCiAgICAgICAgcmVzcF9iID0gcmFuZG9tLmNob2ljZShfUkVTUE9OU0VfVEVNUExBVEVTX0IpLmZvcm1hdCh0b3BpYz10b3BpYykKCiAgICAgICAgbGFiZWwgPSBybmcuY2hvaWNlKFswLCAxLCAyXSwgcD1bMC40LCAwLjQsIDAuMl0pCiAgICAgICAgdHJhaW5fcm93cy5hcHBlbmQoewogICAgICAgICAgICAiaWQiOiAxMDAwMDAgKyBpLAogICAgICAgICAgICAibW9kZWxfYSI6IGYibW9kZWxfeF97cm5nLnJhbmRpbnQoMSwgNSl9IiwKICAgICAgICAgICAgIm1vZGVsX2IiOiBmIm1vZGVsX3lfe3JuZy5yYW5kaW50KDEsIDUpfSIsCiAgICAgICAgICAgICJwcm9tcHQiOiBwcm9tcHQsCiAgICAgICAgICAgICJyZXNwb25zZV9hIjogcmVzcF9hLAogICAgICAgICAgICAicmVzcG9uc2VfYiI6IHJlc3BfYiwKICAgICAgICAgICAgIndpbm5lcl9tb2RlbF9hIjogMSBpZiBsYWJlbCA9PSAwIGVsc2UgMCwKICAgICAgICAgICAgIndpbm5lcl9tb2RlbF9iIjogMSBpZiBsYWJlbCA9PSAxIGVsc2UgMCwKICAgICAgICAgICAgIndpbm5lcl90aWUiOiAxIGlmIGxhYmVsID09IDIgZWxzZSAwLAogICAgICAgIH0pCgogICAgdHJhaW5fcGF0aCA9IG91dCAvICJ0cmFpbi5jc3YiCiAgICBwZC5EYXRhRnJhbWUodHJhaW5fcm93cykudG9fY3N2KHRyYWluX3BhdGgsIGluZGV4PUZhbHNlKQoKICAgICMgLS0tLSDmtYvor5Xpm4YgLS0tLQogICAgdGVzdF9yb3dzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG5fdGVzdCk6CiAgICAgICAgcHJvbXB0ID0gX1BST01QVFNbaSAlIGxlbihfUFJPTVBUUyldCiAgICAgICAgdG9waWMgPSBwcm9tcHQuc3BsaXQoIj8iKVswXS5zcGxpdCgiLiIpWzBdLnN0cmlwKCkKCiAgICAgICAgcmVzcF9hID0gcmFuZG9tLmNob2ljZShfUkVTUE9OU0VfVEVNUExBVEVTX0EpLmZvcm1hdCh0b3BpYz10b3BpYykKICAgICAgICByZXNwX2IgPSByYW5kb20uY2hvaWNlKF9SRVNQT05TRV9URU1QTEFURVNfQikuZm9ybWF0KHRvcGljPXRvcGljKQoKICAgICAgICB0ZXN0X3Jvd3MuYXBwZW5kKHsKICAgICAgICAgICAgImlkIjogMjAwMDAwICsgaSwKICAgICAgICAgICAgInByb21wdCI6IHByb21wdCwKICAgICAgICAgICAgInJlc3BvbnNlX2EiOiByZXNwX2EsCiAgICAgICAgICAgICJyZXNwb25zZV9iIjogcmVzcF9iLAogICAgICAgIH0pCgogICAgdGVzdF9wYXRoID0gb3V0IC8gInRlc3QuY3N2IgogICAgcGQuRGF0YUZyYW1lKHRlc3Rfcm93cykudG9fY3N2KHRlc3RfcGF0aCwgaW5kZXg9RmFsc2UpCgogICAgcHJpbnQoZiLlt7LnlJ/miJDorq3nu4Ppm4Y6IHt0cmFpbl9wYXRofSAoe25fdHJhaW59IOadoSkiKQogICAgcHJpbnQoZiLlt7LnlJ/miJDmtYvor5Xpm4Y6IHt0ZXN0X3BhdGh9ICh7bl90ZXN0fSDmnaEpIikKICAgIHJldHVybiB0cmFpbl9wYXRoLCB0ZXN0X3BhdGgKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0i55Sf5oiQ5YGH5pWw5o2u55So5LqO5pys5Zyw5rWL6K+V44CCIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSIuIiwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6L6T5Ye655uu5b2VIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbi10cmFpbiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMCwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6K6t57uD6ZuG6KGM5pWwIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbi10ZXN0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iua1i+ivlembhuihjOaVsCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgZ2VuZXJhdGVfZHVtbXlfZGF0YSgKICAgICAgICBvdXRwdXRfZGlyPWFyZ3Mub3V0cHV0X2RpciwKICAgICAgICBuX3RyYWluPWFyZ3Mubl90cmFpbiwKICAgICAgICBuX3Rlc3Q9YXJncy5uX3Rlc3QsCiAgICAgICAgc2VlZD1hcmdzLnNlZWQsCiAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "train.py": "IiIiCuiuree7g+iEmuacrCDigJQg5L2/55SoIEh1Z2dpbmdGYWNlIFRyYWluZXIg6L+b6KGMIFFMb1JBIOW+ruiwg+OAggoK5a6M5pW05rWB56iL77yaCiAgMS4g5Yqg6L296YWN572uICjpu5jorqTlgLwgKyBZQU1MIOimhuebliArIENMSSDopobnm5YpCiAgMi4g55Sf5oiQ5oiW5Yqg6L296K6t57uD5pWw5o2uCiAgMy4gVG9rZW5pemF0aW9uIChhcHBseV9jaGF0X3RlbXBsYXRlKQogIDQuIOWKoOi9vSBRd2VuMy41LTAuOEIgKyA0LWJpdCDph4/ljJYgKyBMb1JBCiAgNS4g5L2/55SoIFRyYWluZXIg6K6t57uDCiAgNi4g5L+d5a2YIGFkYXB0ZXIgKyB0b2tlbml6ZXIgKyDphY3nva4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IERhdGFDb2xsYXRvcldpdGhQYWRkaW5nLCBUcmFpbmVyLCBUcmFpbmluZ0FyZ3VtZW50cwoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBBcHBDb25maWcsIGxvYWRfY29uZmlnCmZyb20gYXJlbmFfcmFua2VyLmRhdGEgaW1wb3J0ICgKICAgIGJ1aWxkX2RhdGFzZXQsCiAgICBjb21wdXRlX21ldHJpY3MsCiAgICBsb2FkX2FuZF9wcmVwcm9jZXNzLAogICAgc3BsaXRfdHJhaW5fdmFsaWQsCikKZnJvbSBhcmVuYV9yYW5rZXIuaGYgaW1wb3J0IGxvYWRfbW9kZWwsIGxvYWRfdG9rZW5pemVyCgpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcigiYXJlbmFfcmFua2VyLnRyYWluIikKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDTEkg5Y+C5pWw6Kej5p6QCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgcGFyc2VfYXJncygpIC0+IGFyZ3BhcnNlLk5hbWVzcGFjZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKAogICAgICAgIGRlc2NyaXB0aW9uPSJRTG9SQSBmaW5lLXR1bmUgUXdlbjMuNS0wLjhCIGZvciBBcmVuYSBwcmVmZXJlbmNlIHByZWRpY3Rpb24uIgogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCB0eXBlPXN0ciwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJZQU1MIOmFjee9ruaWh+S7tui3r+W+hCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Ii4iLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJ0cmFpbi5jc3YgLyB0ZXN0LmNzdiDmiYDlnKjnm67lvZUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tb2RlbC1uYW1lIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i5Z+65bqn5qih5Z6L5ZCN56ew5oiW5pys5Zyw6Lev5b6EIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iuiuree7g+S6p+eJqei+k+WHuuebruW9lSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heC1sZW5ndGgiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJ0b2tlbml6ZXIg5pyA5aSn6ZW/5bqmIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6K6t57uDIGVwb2NoIOaVsCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJwZXJfZGV2aWNlX3RyYWluX2JhdGNoX3NpemUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ncmFkLWFjY3VtLXN0ZXBzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i5qKv5bqm57Sv56ev5q2l5pWwIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGVhcm5pbmctcmF0ZSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i5a2m5Lmg546HIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2FjaGUtZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbG9jYWwtZmlsZXMtb25seSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRpc2FibGUtbG9yYSIsIGRlc3Q9InVzZV9sb3JhIiwgYWN0aW9uPSJzdG9yZV9mYWxzZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWxvcmEtciIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWxvcmEtYWxwaGEiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1uby00Yml0IiwgZGVzdD0ibG9hZF9pbl80Yml0IiwgYWN0aW9uPSJzdG9yZV9mYWxzZSIpCiAgICBwYXJzZXIuc2V0X2RlZmF1bHRzKHVzZV9sb3JhPU5vbmUsIGxvYWRfaW5fNGJpdD1Ob25lKQogICAgcmV0dXJuIHBhcnNlci5wYXJzZV9hcmdzKCkKCgpkZWYgYXBwbHlfb3ZlcnJpZGVzKGNvbmZpZzogQXBwQ29uZmlnLCBhcmdzOiBhcmdwYXJzZS5OYW1lc3BhY2UpIC0+IEFwcENvbmZpZzoKICAgICIiIuWwhiBDTEkg5Y+C5pWw6KaG55uW5Yiw6YWN572u5Lit44CCIiIiCiAgICBpZiBhcmdzLm1vZGVsX25hbWU6CiAgICAgICAgY29uZmlnLm1vZGVsLm1vZGVsX25hbWUgPSBhcmdzLm1vZGVsX25hbWUKICAgIGlmIGFyZ3Mub3V0cHV0X2RpcjoKICAgICAgICBjb25maWcudHJhaW5pbmcub3V0cHV0X2RpciA9IGFyZ3Mub3V0cHV0X2RpcgogICAgaWYgYXJncy5tYXhfbGVuZ3RoIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC5tYXhfbGVuZ3RoID0gYXJncy5tYXhfbGVuZ3RoCiAgICBpZiBhcmdzLmVwb2NocyBpcyBub3QgTm9uZToKICAgICAgICBjb25maWcudHJhaW5pbmcubnVtX3RyYWluX2Vwb2NocyA9IGFyZ3MuZXBvY2hzCiAgICBpZiBhcmdzLmJhdGNoX3NpemUgaXMgbm90IE5vbmU6CiAgICAgICAgY29uZmlnLnRyYWluaW5nLnBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZSA9IGFyZ3MuYmF0Y2hfc2l6ZQogICAgaWYgYXJncy5ncmFkX2FjY3VtX3N0ZXBzIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy50cmFpbmluZy5ncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMgPSBhcmdzLmdyYWRfYWNjdW1fc3RlcHMKICAgIGlmIGFyZ3MubGVhcm5pbmdfcmF0ZSBpcyBub3QgTm9uZToKICAgICAgICBjb25maWcudHJhaW5pbmcubGVhcm5pbmdfcmF0ZSA9IGFyZ3MubGVhcm5pbmdfcmF0ZQogICAgaWYgYXJncy5jYWNoZV9kaXIgaXMgbm90IE5vbmU6CiAgICAgICAgY29uZmlnLm1vZGVsLmNhY2hlX2RpciA9IGFyZ3MuY2FjaGVfZGlyCiAgICBpZiBhcmdzLmxvY2FsX2ZpbGVzX29ubHk6CiAgICAgICAgY29uZmlnLm1vZGVsLmxvY2FsX2ZpbGVzX29ubHkgPSBUcnVlCiAgICBpZiBhcmdzLnVzZV9sb3JhIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC51c2VfbG9yYSA9IGFyZ3MudXNlX2xvcmEKICAgIGlmIGFyZ3MubG9yYV9yIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC5sb3JhX3IgPSBhcmdzLmxvcmFfcgogICAgaWYgYXJncy5sb3JhX2FscGhhIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC5sb3JhX2FscGhhID0gYXJncy5sb3JhX2FscGhhCiAgICBpZiBhcmdzLmxvYWRfaW5fNGJpdCBpcyBub3QgTm9uZToKICAgICAgICBjb25maWcubW9kZWwubG9hZF9pbl80Yml0ID0gYXJncy5sb2FkX2luXzRiaXQKICAgIHJldHVybiBjb25maWcKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDlt6Xlhbflh73mlbAKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBzZXRfc2VlZChzZWVkOiBpbnQpIC0+IE5vbmU6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQoKCmRlZiBzZXR1cF9sb2dnaW5nKCkgLT4gTm9uZToKICAgIGxvZ2dpbmcuYmFzaWNDb25maWcoCiAgICAgICAgbGV2ZWw9bG9nZ2luZy5JTkZPLAogICAgICAgIGZvcm1hdD0iJShhc2N0aW1lKXMgfCAlKGxldmVsbmFtZSlzIHwgJShtZXNzYWdlKXMiLAogICAgICAgIGRhdGVmbXQ9IiVIOiVNOiVTIiwKICAgICkKCgpkZWYgZm9ybWF0X3NlY29uZHMoc2Vjb25kczogZmxvYXQpIC0+IHN0cjoKICAgIHRvdGFsID0gbWF4KGludChzZWNvbmRzKSwgMCkKICAgIG0sIHMgPSBkaXZtb2QodG90YWwsIDYwKQogICAgaCwgbSA9IGRpdm1vZChtLCA2MCkKICAgIGlmIGggPiAwOgogICAgICAgIHJldHVybiBmIntofWgge219bSB7c31zIgogICAgaWYgbSA+IDA6CiAgICAgICAgcmV0dXJuIGYie219bSB7c31zIgogICAgcmV0dXJuIGYie3N9cyIKCgpkZWYgZGVzY3JpYmVfZGV2aWNlKGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBzdHI6CiAgICBpZiBkZXZpY2UudHlwZSAhPSAiY3VkYSI6CiAgICAgICAgcmV0dXJuICJDUFUiCiAgICBuYW1lID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoZGV2aWNlKQogICAgbWVtID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoZGV2aWNlKS50b3RhbF9tZW1vcnkgLyAxMDI0KiozCiAgICByZXR1cm4gZiJ7bmFtZX0gKHttZW06LjFmfSBHQikiCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5p6E5bu6IFRyYWluaW5nQXJndW1lbnRzCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgYnVpbGRfdHJhaW5pbmdfYXJncyhjb25maWc6IEFwcENvbmZpZykgLT4gVHJhaW5pbmdBcmd1bWVudHM6CiAgICAiIiLku44gQXBwQ29uZmlnIOaehOW7uiBIdWdnaW5nRmFjZSBUcmFpbmluZ0FyZ3VtZW50c+OAgiIiIgogICAgdGMgPSBjb25maWcudHJhaW5pbmcKICAgIGhhc19jdWRhID0gdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQoKICAgICMgQ1BVIOaooeW8j+S4i+iHquWKqOiwg+aVtOS4jeWFvOWuueeahOWPguaVsAogICAgb3B0aW0gPSB0Yy5vcHRpbQogICAgZnAxNiA9IHRjLmZwMTYKICAgIGJmMTYgPSB0Yy5iZjE2CiAgICBpZiBub3QgaGFzX2N1ZGE6CiAgICAgICAgaWYgb3B0aW0uc3RhcnRzd2l0aCgicGFnZWRfIik6CiAgICAgICAgICAgIG9wdGltID0gImFkYW13X3RvcmNoIgogICAgICAgICAgICBMT0dHRVIud2FybmluZygiQ1BVIOaooeW8jzog5LyY5YyW5Zmo5LuOICVzIOWbnumAgOS4uiBhZGFtd190b3JjaCIsIHRjLm9wdGltKQogICAgICAgIGZwMTYgPSBGYWxzZQogICAgICAgIGJmMTYgPSBGYWxzZQogICAgICAgIExPR0dFUi53YXJuaW5nKCJDUFUg5qih5byPOiDlt7LnpoHnlKggZnAxNi9iZjE2IOa3t+WQiOeyvuW6piIpCgogICAgcmV0dXJuIFRyYWluaW5nQXJndW1lbnRzKAogICAgICAgIG91dHB1dF9kaXI9dGMub3V0cHV0X2RpciwKICAgICAgICBsZWFybmluZ19yYXRlPXRjLmxlYXJuaW5nX3JhdGUsCiAgICAgICAgd2VpZ2h0X2RlY2F5PXRjLndlaWdodF9kZWNheSwKICAgICAgICBwZXJfZGV2aWNlX3RyYWluX2JhdGNoX3NpemU9dGMucGVyX2RldmljZV90cmFpbl9iYXRjaF9zaXplLAogICAgICAgIHBlcl9kZXZpY2VfZXZhbF9iYXRjaF9zaXplPXRjLnBlcl9kZXZpY2VfZXZhbF9iYXRjaF9zaXplLAogICAgICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcz10Yy5ncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMsCiAgICAgICAgbnVtX3RyYWluX2Vwb2Nocz10Yy5udW1fdHJhaW5fZXBvY2hzLAogICAgICAgIHdhcm11cF9zdGVwcz10Yy53YXJtdXBfc3RlcHMsCiAgICAgICAgbHJfc2NoZWR1bGVyX3R5cGU9dGMubHJfc2NoZWR1bGVyX3R5cGUsCiAgICAgICAgb3B0aW09b3B0aW0sCiAgICAgICAgZnAxNj1mcDE2LAogICAgICAgIGJmMTY9YmYxNiwKICAgICAgICBncmFkaWVudF9jaGVja3BvaW50aW5nPXRjLmdyYWRpZW50X2NoZWNrcG9pbnRpbmcsCiAgICAgICAgbG9nZ2luZ19zdGVwcz10Yy5sb2dnaW5nX3N0ZXBzLAogICAgICAgIGV2YWxfc3RyYXRlZ3k9dGMuZXZhbF9zdHJhdGVneSwKICAgICAgICBzYXZlX3N0cmF0ZWd5PXRjLnNhdmVfc3RyYXRlZ3ksCiAgICAgICAgc2F2ZV90b3RhbF9saW1pdD10Yy5zYXZlX3RvdGFsX2xpbWl0LAogICAgICAgIGxvYWRfYmVzdF9tb2RlbF9hdF9lbmQ9dGMubG9hZF9iZXN0X21vZGVsX2F0X2VuZCwKICAgICAgICBtZXRyaWNfZm9yX2Jlc3RfbW9kZWw9dGMubWV0cmljX2Zvcl9iZXN0X21vZGVsLAogICAgICAgIGdyZWF0ZXJfaXNfYmV0dGVyPXRjLmdyZWF0ZXJfaXNfYmV0dGVyLAogICAgICAgIHNlZWQ9dGMuc2VlZCwKICAgICAgICByZXBvcnRfdG89dGMucmVwb3J0X3RvLAogICAgICAgIGRhdGFsb2FkZXJfbnVtX3dvcmtlcnM9dGMuZGF0YWxvYWRlcl9udW1fd29ya2VycywKICAgICAgICAjIFBFRlQgKyBncmFkaWVudCBjaGVja3BvaW50aW5nIOWFvOWuueaApwogICAgICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmdfa3dhcmdzPXsidXNlX3JlZW50cmFudCI6IEZhbHNlfQogICAgICAgIGlmIHRjLmdyYWRpZW50X2NoZWNrcG9pbnRpbmcKICAgICAgICBlbHNlIE5vbmUsCiAgICApCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5pel5b+X5pGY6KaBCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgbG9nX3J1bl9zdW1tYXJ5KAogICAgY29uZmlnOiBBcHBDb25maWcsCiAgICBkZXZpY2U6IHRvcmNoLmRldmljZSwKICAgIHRyYWluX3NpemU6IGludCwKICAgIHZhbGlkX3NpemU6IGludCwKKSAtPiBOb25lOgogICAgbWMgPSBjb25maWcubW9kZWwKICAgIHRjID0gY29uZmlnLnRyYWluaW5nCiAgICBMT0dHRVIuaW5mbygiPSIgKiA2MCkKICAgIExPR0dFUi5pbmZvKCLorq3nu4PlkK/liqgiKQogICAgTE9HR0VSLmluZm8oIuiuvuWkhzogJXMiLCBkZXNjcmliZV9kZXZpY2UoZGV2aWNlKSkKICAgIExPR0dFUi5pbmZvKAogICAgICAgICLmqKHlnos6ICVzIHwgbWF4X2xlbmd0aD0lcyB8IDRiaXQ9JXMgfCBMb1JBPSVzIiwKICAgICAgICBtYy5tb2RlbF9uYW1lLCBtYy5tYXhfbGVuZ3RoLAogICAgICAgICJvbiIgaWYgbWMubG9hZF9pbl80Yml0IGVsc2UgIm9mZiIsCiAgICAgICAgIm9uIiBpZiBtYy51c2VfbG9yYSBlbHNlICJvZmYiLAogICAgKQogICAgaWYgbWMudXNlX2xvcmE6CiAgICAgICAgTE9HR0VSLmluZm8oCiAgICAgICAgICAgICJMb1JBIOmFjee9rjogcj0lcywgYWxwaGE9JXMsIGRyb3BvdXQ9JS4zZiwgbW9kdWxlcz0lcywgbW9kdWxlc190b19zYXZlPSVzIiwKICAgICAgICAgICAgbWMubG9yYV9yLCBtYy5sb3JhX2FscGhhLCBtYy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgICAgICIsIi5qb2luKG1jLmxvcmFfdGFyZ2V0X21vZHVsZXMpLAogICAgICAgICAgICAiLCIuam9pbihtYy5sb3JhX21vZHVsZXNfdG9fc2F2ZSksCiAgICAgICAgKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIuaVsOaNrumbhjogdHJhaW49JXMsIHZhbGlkPSVzIiwKICAgICAgICB0cmFpbl9zaXplLCB2YWxpZF9zaXplLAogICAgKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIuiuree7g+WPguaVsDogbHI9JS4xZSwgZXBvY2hzPSVzLCBiYXRjaD0lcywgZ3JhZF9hY2N1bT0lcywgc2NoZWR1bGVyPSVzLCB3YXJtdXA9JS4yZiIsCiAgICAgICAgdGMubGVhcm5pbmdfcmF0ZSwgdGMubnVtX3RyYWluX2Vwb2NocywKICAgICAgICB0Yy5wZXJfZGV2aWNlX3RyYWluX2JhdGNoX3NpemUsCiAgICAgICAgdGMuZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzLAogICAgICAgIHRjLmxyX3NjaGVkdWxlcl90eXBlLAogICAgICAgIHRjLndhcm11cF9zdGVwcywKICAgICkKICAgIExPR0dFUi5pbmZvKCLovpPlh7rnm67lvZU6ICVzIiwgdGMub3V0cHV0X2RpcikKICAgIExPR0dFUi5pbmZvKCI9IiAqIDYwKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOS/neWtmOiuree7g+S6p+eJqQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIHNhdmVfYXJ0aWZhY3RzKAogICAgb3V0cHV0X2RpcjogUGF0aCwKICAgIG1vZGVsLAogICAgdG9rZW5pemVyLAogICAgY29uZmlnOiBBcHBDb25maWcsCiAgICBtZXRyaWNzOiBkaWN0IHwgTm9uZSA9IE5vbmUsCikgLT4gTm9uZToKICAgICIiIgogICAg5L+d5a2Y6K6t57uD5Lqn54mp77yaCiAgICAgIC0gYWRhcHRlciDmnYPph40gKExvUkEpICsg5YiG57G75aS0CiAgICAgIC0gdG9rZW5pemVyCiAgICAgIC0g6YWN572u5paH5Lu2CiAgICAgIC0g5pyA5L2z5oyH5qCHICjlpoLmnIkpCiAgICAiIiIKICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgICMg5L+d5a2YIGFkYXB0ZXIgKFBFRlQg5qih5Z6LKSDmiJblrozmlbTmqKHlnosKICAgIG1vZGVsLnNhdmVfcHJldHJhaW5lZChvdXRwdXRfZGlyIC8gIm1vZGVsIikKICAgIHRva2VuaXplci5zYXZlX3ByZXRyYWluZWQob3V0cHV0X2RpciAvICJ0b2tlbml6ZXIiKQogICAgY29uZmlnLnNhdmUob3V0cHV0X2RpciAvICJjb25maWcueWFtbCIpCgogICAgaWYgbWV0cmljczoKICAgICAgICAob3V0cHV0X2RpciAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgICAgICBqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UpLAogICAgICAgICAgICBlbmNvZGluZz0idXRmLTgiLAogICAgICAgICkKCiAgICBMT0dHRVIuaW5mbygi6K6t57uD5Lqn54mp5bey5L+d5a2Y5YiwOiAlcyIsIG91dHB1dF9kaXIpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5Li75Ye95pWwCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBzZXR1cF9sb2dnaW5nKCkKICAgIGFyZ3MgPSBwYXJzZV9hcmdzKCkKICAgIGNvbmZpZyA9IGFwcGx5X292ZXJyaWRlcyhsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZyksIGFyZ3MpCiAgICBzZXRfc2VlZChjb25maWcudHJhaW5pbmcuc2VlZCkKCiAgICBkYXRhX2RpciA9IFBhdGgoYXJncy5kYXRhX2RpcikKICAgIG91dHB1dF9kaXIgPSBQYXRoKGNvbmZpZy50cmFpbmluZy5vdXRwdXRfZGlyKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBzdGFydGVkX2F0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQoKICAgICMgLS0tLSAxLiDliqDovb3mlbDmja4gLS0tLQogICAgTE9HR0VSLmluZm8oIuWKoOi9veiuree7g+aVsOaNrjogJXMiLCBkYXRhX2RpciAvIGNvbmZpZy5kYXRhLnRyYWluX3BhdGgpCiAgICB0cmFpbl9kZiA9IGxvYWRfYW5kX3ByZXByb2Nlc3MoCiAgICAgICAgc3RyKGRhdGFfZGlyIC8gY29uZmlnLmRhdGEudHJhaW5fcGF0aCksCiAgICAgICAgbWF4X2NoYXJzPWNvbmZpZy5kYXRhLnRleHRfbWF4X2NoYXJzLAogICAgICAgIGlzX3RyYWluPVRydWUsCiAgICApCiAgICB0cmFpbl9zcGxpdCwgdmFsaWRfc3BsaXQgPSBzcGxpdF90cmFpbl92YWxpZCh0cmFpbl9kZiwgY29uZmlnLmRhdGEpCiAgICBMT0dHRVIuaW5mbygi6K6t57uD6ZuGOiAlcyDmnaEsIOmqjOivgembhjogJXMg5p2hIiwgbGVuKHRyYWluX3NwbGl0KSwgbGVuKHZhbGlkX3NwbGl0KSkKCiAgICAjIC0tLS0gMi4g5Yqg6L29IHRva2VuaXplciAtLS0tCiAgICBMT0dHRVIuaW5mbygi5Yqg6L29IHRva2VuaXplcjogJXMiLCBjb25maWcubW9kZWwubW9kZWxfbmFtZSkKICAgIHRva2VuaXplciA9IGxvYWRfdG9rZW5pemVyKGNvbmZpZy5tb2RlbCkKCiAgICAjIC0tLS0gMy4gVG9rZW5pemUg5pWw5o2u6ZuGIC0tLS0KICAgIExPR0dFUi5pbmZvKCJUb2tlbml6aW5nIOaVsOaNrumbhiAobWF4X2xlbmd0aD0lcykuLi4iLCBjb25maWcubW9kZWwubWF4X2xlbmd0aCkKICAgIHRyYWluX2RhdGFzZXQgPSBidWlsZF9kYXRhc2V0KAogICAgICAgIHRyYWluX3NwbGl0LCB0b2tlbml6ZXIsIGNvbmZpZy5tb2RlbC5tYXhfbGVuZ3RoLCBpc190cmFpbj1UcnVlLAogICAgKQogICAgdmFsaWRfZGF0YXNldCA9IGJ1aWxkX2RhdGFzZXQoCiAgICAgICAgdmFsaWRfc3BsaXQsIHRva2VuaXplciwgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgsIGlzX3RyYWluPVRydWUsCiAgICApCiAgICBMT0dHRVIuaW5mbygKICAgICAgICAiVG9rZW5pemF0aW9uIOWujOaIkDogdHJhaW49JXMsIHZhbGlkPSVzIiwKICAgICAgICBsZW4odHJhaW5fZGF0YXNldCksIGxlbih2YWxpZF9kYXRhc2V0KSwKICAgICkKCiAgICAjIC0tLS0gNC4g5Yqg6L295qih5Z6LIC0tLS0KICAgIExPR0dFUi5pbmZvKCLliqDovb3mqKHlnosgKFFMb1JBKTogJXMiLCBjb25maWcubW9kZWwubW9kZWxfbmFtZSkKICAgIG1vZGVsID0gbG9hZF9tb2RlbChjb25maWcubW9kZWwsIHRva2VuaXplcj10b2tlbml6ZXIpCgogICAgbG9nX3J1bl9zdW1tYXJ5KGNvbmZpZywgZGV2aWNlLCBsZW4odHJhaW5fZGF0YXNldCksIGxlbih2YWxpZF9kYXRhc2V0KSkKCiAgICAjIC0tLS0gNS4g5p6E5bu6IFRyYWluZXIgLS0tLQogICAgdHJhaW5pbmdfYXJncyA9IGJ1aWxkX3RyYWluaW5nX2FyZ3MoY29uZmlnKQogICAgZGF0YV9jb2xsYXRvciA9IERhdGFDb2xsYXRvcldpdGhQYWRkaW5nKAogICAgICAgIHRva2VuaXplcj10b2tlbml6ZXIsCiAgICAgICAgcGFkZGluZz1UcnVlLAogICAgKQoKICAgIHRyYWluZXIgPSBUcmFpbmVyKAogICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgIGFyZ3M9dHJhaW5pbmdfYXJncywKICAgICAgICB0cmFpbl9kYXRhc2V0PXRyYWluX2RhdGFzZXQsCiAgICAgICAgZXZhbF9kYXRhc2V0PXZhbGlkX2RhdGFzZXQsCiAgICAgICAgZGF0YV9jb2xsYXRvcj1kYXRhX2NvbGxhdG9yLAogICAgICAgIGNvbXB1dGVfbWV0cmljcz1jb21wdXRlX21ldHJpY3MsCiAgICApCgogICAgIyAtLS0tIDYuIOiuree7gyAtLS0tCiAgICBMT0dHRVIuaW5mbygi5byA5aeL6K6t57uDLi4uIikKICAgIHRyYWluX3Jlc3VsdCA9IHRyYWluZXIudHJhaW4oKQoKICAgICMgLS0tLSA3LiDor4TkvLAgLS0tLQogICAgTE9HR0VSLmluZm8oIuiuree7g+WujOaIkO+8jOi/kOihjOacgOe7iOivhOS8sC4uLiIpCiAgICBldmFsX3Jlc3VsdCA9IHRyYWluZXIuZXZhbHVhdGUoKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIuacgOe7iOivhOS8sDogbG9nX2xvc3M9JS40ZiwgYWNjdXJhY3k9JS40ZiIsCiAgICAgICAgZXZhbF9yZXN1bHQuZ2V0KCJldmFsX2xvZ19sb3NzIiwgZmxvYXQoIm5hbiIpKSwKICAgICAgICBldmFsX3Jlc3VsdC5nZXQoImV2YWxfYWNjdXJhY3kiLCBmbG9hdCgibmFuIikpLAogICAgKQoKICAgICMgLS0tLSA4LiDkv53lrZggLS0tLQogICAgbWV0cmljcyA9IHsKICAgICAgICAidHJhaW5fbG9zcyI6IHRyYWluX3Jlc3VsdC50cmFpbmluZ19sb3NzLAogICAgICAgICJldmFsX2xvZ19sb3NzIjogZXZhbF9yZXN1bHQuZ2V0KCJldmFsX2xvZ19sb3NzIiksCiAgICAgICAgImV2YWxfYWNjdXJhY3kiOiBldmFsX3Jlc3VsdC5nZXQoImV2YWxfYWNjdXJhY3kiKSwKICAgIH0KICAgIHNhdmVfYXJ0aWZhY3RzKG91dHB1dF9kaXIsIG1vZGVsLCB0b2tlbml6ZXIsIGNvbmZpZywgbWV0cmljcykKCiAgICBlbGFwc2VkID0gZm9ybWF0X3NlY29uZHModGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWRfYXQpCiAgICBMT0dHRVIuaW5mbygi5YWo6YOo5a6M5oiQ77yM5oC76ICX5pe2OiAlcyIsIGVsYXBzZWQpCiAgICBwcmludChqc29uLmR1bXBzKG1ldHJpY3MsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
    "predict.py": "IiIiCuaOqOeQhuiEmuacrCDigJQg5Yqg6L296K6t57uD5aW955qEIFFMb1JBIGFkYXB0ZXLvvIzlr7nmtYvor5Xpm4bnlJ/miJAgc3VibWlzc2lvbi5jc3bjgIIKCua1geeoi++8mgogIDEuIOmHjeaWsOWKoOi9veWfuuW6p+aooeWeiyAoNC1iaXQg6YeP5YyWKQogIDIuIOWKoOi9veS/neWtmOeahCBMb1JBIGFkYXB0ZXIgKyDliIbnsbvlpLQKICAzLiDpgY3ljobmtYvor5Xpm4bvvIznlJ/miJDkuInliIbnsbvmpoLnjocKICA0LiDmpoLnjofoo4HliaogKyDlvZLkuIDljJblkI7lhpnlhaUgc3VibWlzc2lvbi5jc3YKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCmZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCAoCiAgICBBdXRvQ29uZmlnLAogICAgQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbiwKICAgIEF1dG9Ub2tlbml6ZXIsCiAgICBCaXRzQW5kQnl0ZXNDb25maWcsCiAgICBEYXRhQ29sbGF0b3JXaXRoUGFkZGluZywKKQoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBJRF9UT19MQUJFTCwgTlVNX0xBQkVMUywgbG9hZF9jb25maWcKZnJvbSBhcmVuYV9yYW5rZXIuZGF0YSBpbXBvcnQgYnVpbGRfZGF0YXNldCwgbG9hZF9hbmRfcHJlcHJvY2VzcwoKTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyZW5hX3Jhbmtlci5wcmVkaWN0IikKUFJPQkFCSUxJVFlfRVBTSUxPTiA9IDFlLTYKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDTEkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBwYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IkdlbmVyYXRlIHN1Ym1pc3Npb24gd2l0aCB0cmFpbmVkIFFMb1JBIEFyZW5hIHJhbmtlci4iCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQtZGlyIiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iuiuree7g+S6p+eJqeebruW9lSAo5YyF5ZCrIG1vZGVsLywgdG9rZW5pemVyLywgY29uZmlnLnlhbWwpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD0iLiIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9InRlc3QuY3N2IOaJgOWcqOebruW9lSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1wYXRoIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6L6T5Ye6IHN1Ym1pc3Npb24uY3N2IOi3r+W+hCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD00LAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSLmjqjnkIYgYmF0Y2ggc2l6ZSIpCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOW3peWFt+WHveaVsAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIHNldHVwX2xvZ2dpbmcoKSAtPiBOb25lOgogICAgbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgICAgICBsZXZlbD1sb2dnaW5nLklORk8sCiAgICAgICAgZm9ybWF0PSIlKGFzY3RpbWUpcyB8ICUobGV2ZWxuYW1lKXMgfCAlKG1lc3NhZ2UpcyIsCiAgICAgICAgZGF0ZWZtdD0iJUg6JU06JVMiLAogICAgKQoKCmRlZiBmb3JtYXRfc2Vjb25kcyhzZWNvbmRzOiBmbG9hdCkgLT4gc3RyOgogICAgdG90YWwgPSBtYXgoaW50KHNlY29uZHMpLCAwKQogICAgbSwgcyA9IGRpdm1vZCh0b3RhbCwgNjApCiAgICBoLCBtID0gZGl2bW9kKG0sIDYwKQogICAgaWYgaCA+IDA6CiAgICAgICAgcmV0dXJuIGYie2h9aCB7bX1tIHtzfXMiCiAgICBpZiBtID4gMDoKICAgICAgICByZXR1cm4gZiJ7bX1tIHtzfXMiCiAgICByZXR1cm4gZiJ7c31zIgoKCmRlZiBkZXNjcmliZV9kZXZpY2UoZGV2aWNlOiB0b3JjaC5kZXZpY2UpIC0+IHN0cjoKICAgIGlmIGRldmljZS50eXBlICE9ICJjdWRhIjoKICAgICAgICByZXR1cm4gIkNQVSIKICAgIG5hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShkZXZpY2UpCiAgICBtZW0gPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhkZXZpY2UpLnRvdGFsX21lbW9yeSAvIDEwMjQqKjMKICAgIHJldHVybiBmIntuYW1lfSAoe21lbTouMWZ9IEdCKSIKCgpkZWYgbm9ybWFsaXplX3Byb2JhYmlsaXRpZXMoCiAgICBsb2dpdHM6IHRvcmNoLlRlbnNvciwKICAgIGVwc2lsb246IGZsb2F0ID0gUFJPQkFCSUxJVFlfRVBTSUxPTiwKKSAtPiBucC5uZGFycmF5OgogICAgIiIibG9naXRzIOKGkiDoo4HliarlkI7nmoTmpoLnjofvvIjnoa7kv50gbG9nX2xvc3Mg5LiN5Lya54iG54K477yJ44CCIiIiCiAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgobG9naXRzLCBkaW09LTEpLmNwdSgpLm51bXB5KCkKICAgIHByb2JzID0gbnAuY2xpcChwcm9icywgZXBzaWxvbiwgMS4wIC0gZXBzaWxvbikKICAgIHByb2JzID0gcHJvYnMgLyBwcm9icy5zdW0oYXhpcz0tMSwga2VlcGRpbXM9VHJ1ZSkKICAgIHJldHVybiBwcm9icwoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOWKoOi9veaOqOeQhuaooeWeiwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIGxvYWRfaW5mZXJlbmNlX21vZGVsKGNoZWNrcG9pbnRfZGlyOiBQYXRoLCBjb25maWcsIHRva2VuaXplcik6CiAgICAiIiIKICAgIOWKoOi9veiuree7g+WlveeahCBRTG9SQSDmqKHlnovnlKjkuo7mjqjnkIbjgIIKCiAgICDmraXpqqTvvJoKICAgICAgMS4g6YeN5paw5Yqg6L295Z+65bqn5qih5Z6LICg0LWJpdCDph4/ljJYpCiAgICAgIDIuIOS7jiBhZGFwdGVyIOebruW9leWKoOi9vSBMb1JBIOadg+mHjSArIHNjb3JlIOWIhuexu+WktAogICAgICAzLiDlr7npvZAgcGFkX3Rva2VuX2lkCiAgICAiIiIKICAgIG1jID0gY29uZmlnLm1vZGVsCiAgICBhZGFwdGVyX2RpciA9IGNoZWNrcG9pbnRfZGlyIC8gIm1vZGVsIgoKICAgICMg6YeP5YyW6YWN572u77yINC1iaXQg6ZyA6KaBIENVREHvvIkKICAgIGJuYl9jb25maWcgPSBOb25lCiAgICBpZiBtYy5sb2FkX2luXzRiaXQgYW5kIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgYm5iX2NvbmZpZyA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICAgICAgbG9hZF9pbl80Yml0PVRydWUsCiAgICAgICAgICAgIGJuYl80Yml0X3F1YW50X3R5cGU9bWMuYm5iXzRiaXRfcXVhbnRfdHlwZSwKICAgICAgICAgICAgYm5iXzRiaXRfY29tcHV0ZV9kdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgICAgICBibmJfNGJpdF91c2VfZG91YmxlX3F1YW50PW1jLmJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQsCiAgICAgICAgKQogICAgZWxpZiBtYy5sb2FkX2luXzRiaXQ6CiAgICAgICAgTE9HR0VSLndhcm5pbmcoIuacquajgOa1i+WIsCBDVURB77yM6Lez6L+HIDQtYml0IOmHj+WMliIpCgogICAgIyDliqDovb3phY3nva7lubblpITnkIYgVkxNIG51bV9sYWJlbHMg5Lyg5pKtCiAgICBtb2RlbF9jb25maWcgPSBBdXRvQ29uZmlnLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBtYy5tb2RlbF9uYW1lLAogICAgICAgIG51bV9sYWJlbHM9TlVNX0xBQkVMUywKICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgIGNhY2hlX2Rpcj1tYy5jYWNoZV9kaXIsCiAgICAgICAgbG9jYWxfZmlsZXNfb25seT1tYy5sb2NhbF9maWxlc19vbmx5LAogICAgKQogICAgaWYgaGFzYXR0cihtb2RlbF9jb25maWcsICJ0ZXh0X2NvbmZpZyIpOgogICAgICAgIG1vZGVsX2NvbmZpZy50ZXh0X2NvbmZpZy5udW1fbGFiZWxzID0gTlVNX0xBQkVMUwoKICAgICMg5a+56b2QIHBhZF90b2tlbl9pZO+8iOWIhuexu+aooeWei+mcgOimge+8iQogICAgaWYgdG9rZW5pemVyLnBhZF90b2tlbl9pZCBpcyBub3QgTm9uZToKICAgICAgICBtb2RlbF9jb25maWcucGFkX3Rva2VuX2lkID0gdG9rZW5pemVyLnBhZF90b2tlbl9pZAogICAgICAgIGlmIGhhc2F0dHIobW9kZWxfY29uZmlnLCAidGV4dF9jb25maWciKToKICAgICAgICAgICAgbW9kZWxfY29uZmlnLnRleHRfY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKCiAgICAjIOWKoOi9veWfuuW6p+aooeWeiwogICAgZHR5cGUgPSB0b3JjaC5mbG9hdDE2IGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSB0b3JjaC5mbG9hdDMyCiAgICBiYXNlX21vZGVsID0gQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbi5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgbWMubW9kZWxfbmFtZSwKICAgICAgICBjb25maWc9bW9kZWxfY29uZmlnLAogICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Ym5iX2NvbmZpZywKICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgIGNhY2hlX2Rpcj1tYy5jYWNoZV9kaXIsCiAgICAgICAgbG9jYWxfZmlsZXNfb25seT1tYy5sb2NhbF9maWxlc19vbmx5LAogICAgICAgIHRvcmNoX2R0eXBlPWR0eXBlLAogICAgKQoKICAgICMg5Yqg6L29IExvUkEgYWRhcHRlcgogICAgaWYgbWMudXNlX2xvcmEgYW5kIGFkYXB0ZXJfZGlyLmV4aXN0cygpOgogICAgICAgIG1vZGVsID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChiYXNlX21vZGVsLCBzdHIoYWRhcHRlcl9kaXIpKQogICAgICAgIExPR0dFUi5pbmZvKCLlt7LliqDovb0gTG9SQSBhZGFwdGVyOiAlcyIsIGFkYXB0ZXJfZGlyKQogICAgZWxzZToKICAgICAgICBtb2RlbCA9IGJhc2VfbW9kZWwKICAgICAgICBMT0dHRVIuaW5mbygi5pyq5L2/55SoIExvUkEgYWRhcHRlcu+8jOWKoOi9veWujOaVtOaooeWeiyIpCgogICAgbW9kZWwuZXZhbCgpCiAgICByZXR1cm4gbW9kZWwKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDkuLvlh73mlbAKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHNldHVwX2xvZ2dpbmcoKQogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQogICAgY2hlY2twb2ludF9kaXIgPSBQYXRoKGFyZ3MuY2hlY2twb2ludF9kaXIpCiAgICBzdGFydGVkX2F0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQoKICAgICMgLS0tLSAxLiDliqDovb3phY3nva4gLS0tLQogICAgY29uZmlnX3BhdGggPSBjaGVja3BvaW50X2RpciAvICJjb25maWcueWFtbCIKICAgIGlmIG5vdCBjb25maWdfcGF0aC5leGlzdHMoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiLnvLrlsJHphY3nva7mlofku7Y6IHtjb25maWdfcGF0aH0iKQogICAgY29uZmlnID0gbG9hZF9jb25maWcoY29uZmlnX3BhdGgpCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBMT0dHRVIuaW5mbygi6aKE5rWL5ZCv5YqoIikKICAgIExPR0dFUi5pbmZvKCJjaGVja3BvaW50OiAlcyIsIGNoZWNrcG9pbnRfZGlyKQogICAgTE9HR0VSLmluZm8oIuiuvuWkhzogJXMiLCBkZXNjcmliZV9kZXZpY2UoZGV2aWNlKSkKCiAgICAjIC0tLS0gMi4g5Yqg6L29IHRva2VuaXplciAtLS0tCiAgICB0b2tlbml6ZXJfZGlyID0gY2hlY2twb2ludF9kaXIgLyAidG9rZW5pemVyIgogICAgaWYgbm90IHRva2VuaXplcl9kaXIuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYi57y65bCRIHRva2VuaXplciDnm67lvZU6IHt0b2tlbml6ZXJfZGlyfSIpCiAgICB0b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBzdHIodG9rZW5pemVyX2RpciksIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICApCiAgICBpZiB0b2tlbml6ZXIucGFkX3Rva2VuIGlzIE5vbmU6CiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbiA9IHRva2VuaXplci5lb3NfdG9rZW4KICAgICAgICB0b2tlbml6ZXIucGFkX3Rva2VuX2lkID0gdG9rZW5pemVyLmVvc190b2tlbl9pZAogICAgdG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IgoKICAgICMgLS0tLSAzLiDliqDovb3mqKHlnosgLS0tLQogICAgbW9kZWwgPSBsb2FkX2luZmVyZW5jZV9tb2RlbChjaGVja3BvaW50X2RpciwgY29uZmlnLCB0b2tlbml6ZXIpCgogICAgIyAtLS0tIDQuIOWKoOi9vea1i+ivleaVsOaNriAtLS0tCiAgICB0ZXN0X2NzdiA9IFBhdGgoYXJncy5kYXRhX2RpcikgLyBjb25maWcuZGF0YS50ZXN0X3BhdGgKICAgIExPR0dFUi5pbmZvKCLliqDovb3mtYvor5XmlbDmja46ICVzIiwgdGVzdF9jc3YpCiAgICB0ZXN0X2RmID0gbG9hZF9hbmRfcHJlcHJvY2VzcygKICAgICAgICBzdHIodGVzdF9jc3YpLAogICAgICAgIG1heF9jaGFycz1jb25maWcuZGF0YS50ZXh0X21heF9jaGFycywKICAgICAgICBpc190cmFpbj1GYWxzZSwKICAgICkKICAgIHRlc3RfaWRzID0gdGVzdF9kZlsiaWQiXS50b2xpc3QoKQoKICAgIHRlc3RfZGF0YXNldCA9IGJ1aWxkX2RhdGFzZXQoCiAgICAgICAgdGVzdF9kZiwgdG9rZW5pemVyLCBjb25maWcubW9kZWwubWF4X2xlbmd0aCwgaXNfdHJhaW49RmFsc2UsCiAgICApCiAgICBMT0dHRVIuaW5mbygi5b6F6aKE5rWL5qC35pysOiAlcyIsIGxlbih0ZXN0X2RhdGFzZXQpKQoKICAgICMgLS0tLSA1LiDmjqjnkIYgLS0tLQogICAgZGF0YV9jb2xsYXRvciA9IERhdGFDb2xsYXRvcldpdGhQYWRkaW5nKHRva2VuaXplcj10b2tlbml6ZXIsIHBhZGRpbmc9VHJ1ZSkKICAgIGxvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgdGVzdF9kYXRhc2V0LAogICAgICAgIGJhdGNoX3NpemU9YXJncy5iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9RmFsc2UsCiAgICAgICAgY29sbGF0ZV9mbj1kYXRhX2NvbGxhdG9yLAogICAgKQoKICAgIGFsbF9wcm9icyA9IFtdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgYmF0Y2ggaW4gdHFkbShsb2FkZXIsIGRlc2M9InByZWRpY3QiKToKICAgICAgICAgICAgYmF0Y2ggPSB7azogdi50byhkZXZpY2UpIGZvciBrLCB2IGluIGJhdGNoLml0ZW1zKCl9CiAgICAgICAgICAgIG91dHB1dHMgPSBtb2RlbCgqKmJhdGNoKQogICAgICAgICAgICBwcm9icyA9IG5vcm1hbGl6ZV9wcm9iYWJpbGl0aWVzKG91dHB1dHMubG9naXRzKQogICAgICAgICAgICBhbGxfcHJvYnMuYXBwZW5kKHByb2JzKQoKICAgIGFsbF9wcm9icyA9IG5wLmNvbmNhdGVuYXRlKGFsbF9wcm9icywgYXhpcz0wKQoKICAgICMgLS0tLSA2LiDlhpnlh7ogc3VibWlzc2lvbi5jc3YgLS0tLQogICAgcm93cyA9IFtdCiAgICBmb3Igc2FtcGxlX2lkLCBwcm9iIGluIHppcCh0ZXN0X2lkcywgYWxsX3Byb2JzLCBzdHJpY3Q9VHJ1ZSk6CiAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAiaWQiOiBzYW1wbGVfaWQsCiAgICAgICAgICAgIElEX1RPX0xBQkVMWzBdOiBwcm9iWzBdLAogICAgICAgICAgICBJRF9UT19MQUJFTFsxXTogcHJvYlsxXSwKICAgICAgICAgICAgSURfVE9fTEFCRUxbMl06IHByb2JbMl0sCiAgICAgICAgfSkKCiAgICBvdXRwdXRfcGF0aCA9ICgKICAgICAgICBQYXRoKGFyZ3Mub3V0cHV0X3BhdGgpCiAgICAgICAgaWYgYXJncy5vdXRwdXRfcGF0aAogICAgICAgIGVsc2UgY2hlY2twb2ludF9kaXIgLyAic3VibWlzc2lvbi5jc3YiCiAgICApCiAgICBwZC5EYXRhRnJhbWUocm93cykudG9fY3N2KG91dHB1dF9wYXRoLCBpbmRleD1GYWxzZSkKICAgIExPR0dFUi5pbmZvKAogICAgICAgICLpooTmtYvlrozmiJA6ICVzIOihjCwg6ICX5pe2PSVzIiwKICAgICAgICBsZW4ocm93cyksIGZvcm1hdF9zZWNvbmRzKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkX2F0KSwKICAgICkKICAgIExPR0dFUi5pbmZvKCJzdWJtaXNzaW9uIOW3suS/neWtmOWIsDogJXMiLCBvdXRwdXRfcGF0aCkKICAgIHByaW50KGYic2F2ZWQgc3VibWlzc2lvbiB0byB7b3V0cHV0X3BhdGh9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
}

for _name, _b64 in _FILES.items():
    (PKG_DIR / _name).write_text(
        base64.b64decode(_b64).decode('utf-8'),
        encoding='utf-8',
    )
    print(f"  写入 {_name}")

import sys
if str(PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_DIR.parent))

print("\narena_ranker 已写入:", PKG_DIR)
print("sys.path 已更新")

  写入 __init__.py
  写入 config.py
  写入 data.py
  写入 hf.py
  写入 modeling.py
  写入 dummy_data.py
  写入 train.py
  写入 predict.py

arena_ranker 已写入: /kaggle/working/arena_ranker
sys.path 已更新


In [3]:
import arena_ranker
print('导入成功:', arena_ranker.__file__)

导入成功: /kaggle/working/arena_ranker/__init__.py


## 3. 配置

设置竞赛数据路径和训练参数。

⚠️ **请根据你的实际竞赛修改 `COMPETITION_SLUG`。**

T4 和 P100 都是 16GB 显存，对 0.8B 模型 VRAM 参数相同；
P100 没有 Tensor Core，训练速度会慢一些，但不需要改参数。

In [4]:
import torch

# ============================================================
# 🔧 根据你的情况修改以下参数
# ============================================================
COMPETITION_SLUG = "llm-classification-finetuning"   # ← 改成你的竞赛 slug
MODEL_NAME       = "Qwen/Qwen3-0.6B"              # 基座模型
EPOCHS           = 3
BATCH_SIZE       = 2        # T4/P100 16GB 推荐 2
GRAD_ACCUM_STEPS = 8
MAX_LENGTH       = 1024     # T4/P100 16GB 推荐 1024
LEARNING_RATE    = 2e-4
USE_LORA         = True
LOAD_IN_4BIT     = True     # QLoRA 4-bit 量化
# ============================================================

DATA_DIR    = f"/kaggle/input/datasets/dannyatkaggle/llm-finetune-test1/"
OUTPUT_DIR  = "/kaggle/working/artifacts/default"

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"设备: {device_name} ({vram_gb:.1f} GB)")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")


设备: Tesla P100-PCIE-16GB (15.9 GB)
数据目录: /kaggle/input/datasets/dannyatkaggle/llm-finetune-test1/
输出目录: /kaggle/working/artifacts/default


In [5]:
import os

data_files = os.listdir(DATA_DIR)
print("竞赛数据文件:", data_files)
assert "train.csv" in data_files, f"找不到 train.csv，请检查 COMPETITION_SLUG。当前目录: {DATA_DIR}"
assert "test.csv"  in data_files, f"找不到 test.csv，请检查 COMPETITION_SLUG。当前目录: {DATA_DIR}"
print("✅ 数据验证通过")


竞赛数据文件: ['sample_submission.csv', 'train.csv', 'test.csv']
✅ 数据验证通过


## 4. 训练

使用 QLoRA 微调 `Qwen3.5-0.8B`，通过 HuggingFace Trainer 训练。

| 参数 | T4 / P100 16GB | 8GB 显存 | 说明 |
| --- | --- | --- | --- |
| `BATCH_SIZE` | 2 | 1 | per device |
| `GRAD_ACCUM_STEPS` | 8 | 16 | 有效 batch = BATCH_SIZE × steps |
| `MAX_LENGTH` | 1024 | 512 | 输入序列最大 token 数 |
| `EPOCHS` | 3 | 3 | 训练轮数 |
| `LEARNING_RATE` | 2e-4 | 2e-4 | AdamW 学习率 |
| `LOAD_IN_4BIT` | True | True | 4-bit NF4 量化 (QLoRA) |

> **P100 注意事项**: P100 (Pascal) 没有 Tensor Core，FP16 矩阵运算比 T4 慢，
> 但 bitsandbytes 4-bit 量化和 FP16 混合精度均可正常使用。
> VRAM 参数与 T4 相同，主要差异体现在训练速度上。**不支持 bf16**（已是默认关闭）。

In [6]:
import sys

sys.argv = [
    "arena-train",
    "--data-dir",        DATA_DIR,
    "--output-dir",      OUTPUT_DIR,
    "--model-name",      MODEL_NAME,
    "--epochs",          str(EPOCHS),
    "--batch-size",      str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--max-length",      str(MAX_LENGTH),
    "--learning-rate",   str(LEARNING_RATE),
]

if not USE_LORA:
    sys.argv.append("--disable-lora")
if not LOAD_IN_4BIT:
    sys.argv.append("--no-4bit")

from arena_ranker.train import main as train_main
train_main()


07:51:37 | INFO | 加载训练数据: /kaggle/input/datasets/dannyatkaggle/llm-finetune-test1/train.csv
07:51:38 | INFO | 训练集: 107 条, 验证集: 12 条
07:51:38 | INFO | 加载 tokenizer: Qwen/Qwen3-0.6B
07:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:51:38 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

07:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
07:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json "HTTP/1.1 200 OK"
07:51:38 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

07:51:38 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
07:51:38 | INFO | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
07:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
07:51:38 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/vocab.json "HTTP/1.1 200 OK"
07:51:38 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/vocab.json "HTTP/1.1 200 OK"


...

trainable params: 20,188,160 || all params: 616,241,152 || trainable%: 3.2760


Epoch,Training Loss,Validation Loss,Log Loss,Accuracy,Runtime,Samples Per Second,Steps Per Second
1,No log,4.191103,4.191496,0.250000,2.831700,4.238000,1.059000
2,No log,2.645905,2.645389,0.250000,2.826200,4.246000,1.061000
3,No log,2.424174,2.424256,0.250000,2.826300,4.246000,1.061000


07:53:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:53:29 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:53:29 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:53:29 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:54:55 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:54:55 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:54:55 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/res

07:56:27 | INFO | 最终评估: log_loss=2.4243, accuracy=0.2500
07:56:27 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:56:27 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:56:27 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:56:27 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:56:28 | INFO | 训练产物已保存到: /kaggle/working/artifacts/default
07:56:28 | INFO | 全部完成，总耗时: 4m 50s


{
  "train_loss": 15.648780459449405,
  "eval_log_loss": 2.4242556582107704,
  "eval_accuracy": 0.25
}


## 5. 推理与生成提交文件

加载训练好的 QLoRA adapter + 分类头，对 `test.csv` 推理生成 `submission.csv`。

In [7]:
import importlib, sys

SUBMISSION_PATH = "/kaggle/working/submission.csv"

sys.argv = [
    "arena-predict",
    "--checkpoint-dir", OUTPUT_DIR,
    "--data-dir",       DATA_DIR,
    "--output-path",    SUBMISSION_PATH,
    "--batch-size",     "4",
]

from arena_ranker.predict import main as predict_main
predict_main()


07:56:28 | INFO | 预测启动
07:56:28 | INFO | checkpoint: /kaggle/working/artifacts/default
07:56:28 | INFO | 设备: Tesla P100-PCIE-16GB (15.9 GB)
07:56:30 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:56:30 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
07:56:30 | INFO | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07:56:30 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
07:56:32 | INFO | 已加载 LoRA adapter: /kaggle/working/artifacts/default/model
07:56:32 | INFO | 加载测试数据: /kaggle/input/datasets/dannyatkaggle/llm-finetune-test1/test.csv


Tokenizing:   0%|          | 0/3 [00:00<?, ? examples/s]

07:56:34 | INFO | 待预测样本: 3


predict:   0%|          | 0/1 [00:00<?, ?it/s]

07:56:35 | INFO | 预测完成: 3 行, 耗时=7s
07:56:35 | INFO | submission 已保存到: /kaggle/working/submission.csv


saved submission to /kaggle/working/submission.csv


## 6. 检查提交文件

In [8]:
import pandas as pd

sub = pd.read_csv(SUBMISSION_PATH)
print(f"行数: {len(sub)}")
print(f"列名: {list(sub.columns)}")
print()
print(sub.head(10))
print()
print("各列概率统计:")
print(sub.describe())
print()

row_sums = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print(f"概率行和范围: [{row_sums.min():.6f}, {row_sums.max():.6f}]")
print("✅ submission.csv 已生成:", SUBMISSION_PATH)


行数: 3
列名: ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']

        id  winner_model_a  winner_model_b  winner_tie
0   136060          0.3252        0.560000     0.11480
1   211333          0.1884        0.716300     0.09534
2  1233961          0.6190        0.000024     0.38090

各列概率统计:
                 id  winner_model_a  winner_model_b  winner_tie
count  3.000000e+00        3.000000        3.000000    3.000000
mean   5.271180e+05        0.377533        0.425441    0.197013
std    6.132999e+05        0.220019        0.376619    0.159547
min    1.360600e+05        0.188400        0.000024    0.095340
25%    1.736965e+05        0.256800        0.280012    0.105070
50%    2.113330e+05        0.325200        0.560000    0.114800
75%    7.226470e+05        0.472100        0.638150    0.247850
max    1.233961e+06        0.619000        0.716300    0.380900

概率行和范围: [0.999924, 1.000040]
✅ submission.csv 已生成: /kaggle/working/submission.csv


## 附录 A：离线模式（无需联网）

如果 notebook 不能联网（例如最终提交时），需要提前将模型上传到 Kaggle。

### 步骤

1. **上传模型到 Kaggle**
   - 在本地下载好 `Qwen/Qwen3.5-0.8B` 的完整文件
   - 前往 [kaggle.com/models](https://kaggle.com/models) → New Model
   - 上传模型文件夹（包含 config.json, model.safetensors 等）
   - 或者使用 Kaggle Datasets 上传也可以

2. **在 notebook 中添加模型数据集**
   - 右侧 Add Input → 搜索你上传的模型

3. **修改配置**
   ```python
   MODEL_NAME = "/kaggle/input/<model-dataset-slug>"  # 改为本地路径
   ```

4. **训练时加上 `--local-files-only`**
   ```python
   sys.argv.append("--local-files-only")
   ```


## 附录 B：将代码上传为 Kaggle Dataset

如果不想在 notebook 里内联代码，可以把仓库上传为 Kaggle Dataset：

1. 打包源码：
   ```bash
   zip -r arena-ranker-code.zip src/ pyproject.toml README.md
   ```

2. 上传到 Kaggle Datasets

3. 在 notebook 中安装：
   ```python
   !pip install /kaggle/input/arena-ranker-code/
   ```

4. 使用 CLI 命令：
   ```python
   !arena-train --data-dir /kaggle/input/<slug>/ --output-dir /kaggle/working/artifacts/default
   !arena-predict --checkpoint-dir /kaggle/working/artifacts/default --data-dir /kaggle/input/<slug>/ --output-path /kaggle/working/submission.csv
   ```


## 附录 C：显存不足时的参数调整

如果遇到 OOM，优先按以下顺序调整：

1. 降低 `MAX_LENGTH`（如 512）
2. 降低 `BATCH_SIZE` 到 1
3. 提高 `GRAD_ACCUM_STEPS`
4. 确保 `LOAD_IN_4BIT = True`

```python
# 8GB 显存推荐参数
BATCH_SIZE   = 1
GRAD_ACCUM   = 16
MAX_LENGTH   = 512
LOAD_IN_4BIT = True
```